### Version: (1,1,2,2), heavy transform - peak 78.5%


In [ ]:
# Code: 


YOUR_MEAN = 0.1156
YOUR_STD = 0.2198


train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(12),
    transforms.RandomAffine(degrees=0, translate=(0.05,0.05), scale=(0.95,1.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[YOUR_MEAN], std=[YOUR_STD])
])


test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1), 
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[YOUR_MEAN], std=[YOUR_STD])
])


# load data
train_dataset = datasets.ImageFolder(root="data/ADNI/AD_NC/train", transform=train_transform)
test_dataset  = datasets.ImageFolder(root="data/ADNI/AD_NC/test",  transform=test_transform)

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=1)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=1)

print(f"Train length: {len(train_dataset)}, Test length: {len(test_dataset)}")
print(train_dataset[0][0].mean(), train_dataset[0][0].std())


class SmallBlock(nn.Module):
    def __init__(self, dim, layer_scale_init_value=1e-5):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim)
        # We'll use channels_last LayerNorm via explicit permute
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.pw1 = nn.Linear(dim, 4*dim)
        self.act = nn.GELU()
        self.pw2 = nn.Linear(4*dim, dim)
        if layer_scale_init_value > 0:
            self.gamma = nn.Parameter(layer_scale_init_value * torch.ones((dim)), requires_grad=True)
        else:
            self.gamma = None

    def forward(self, x):
        shortcut = x
        x = self.dwconv(x)                          # N,C,H,W
        x = x.permute(0, 2, 3, 1)                   # N,H,W,C
        x = self.norm(x)
        x = self.pw1(x)
        x = self.act(x)
        x = self.pw2(x)
        if self.gamma is not None:
            x = self.gamma * x
        x = x.permute(0, 3, 1, 2)                   # N,C,H,W
        return shortcut + x

class MiniConvNeXt(nn.Module):
    def __init__(self, in_chans=1, num_classes=2,
                 depths=(1,1,2,2), dims=(32,64,128,256), layer_scale_init_value=1e-5):
        super().__init__()
        assert len(depths)==4 and len(dims)==4

        self.dropout = nn.Dropout(p=0.25)
        # stem
        self.downsamples = nn.ModuleList()
        stem = nn.Sequential(
            nn.Conv2d(in_chans, dims[0], kernel_size=4, stride=4),
            # Use LayerNorm over channels_first by wrapping below in forward_features
        )
        self.downsamples.append(stem)

        for i in range(3):
            self.downsamples.append(
                nn.Sequential(
                    nn.Conv2d(dims[i], dims[i+1], kernel_size=2, stride=2)
                )
            )

        # stages
        self.stages = nn.ModuleList()
        for i in range(4):
            blocks = []
            for _ in range(depths[i]):
                blocks.append(SmallBlock(dims[i], layer_scale_init_value=layer_scale_init_value))
            self.stages.append(nn.Sequential(*blocks))

        # final norm and head
        self.final_norm = nn.LayerNorm(dims[-1], eps=1e-6)
        self.head = nn.Linear(dims[-1], num_classes)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                trunc_normal_(m.weight, std=.02)
                if getattr(m, "bias", None) is not None:
                    nn.init.constant_(m.bias, 0)

    def forward_features(self, x):
        # x: N, C, H, W  (C=1)
        for i in range(4):
            x = self.downsamples[i](x)   # conv downsample (N,C,H,W)
            # pass each stage
            x = self.stages[i](x)
        # global pool
        x = x.mean([-2, -1])            # N, C
        x = self.final_norm(x)
        return x

    def forward(self, x):
        x = self.forward_features(x)
        x = self.dropout(x)
        x = self.head(x)
        return x

# set up model and parameters


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not torch.cuda.is_available():
    print("Warning CUDA not Found. Using CPU")    
model = MiniConvNeXt(in_chans=1, num_classes=2,
                 depths=(1,1,2,2), dims=(32,64,128,256)).to(device)
    
    
EPOCHS = 200
    
criterion = nn.CrossEntropyLoss()
#optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)  # AdamW is preferable
# One-cycle LR often helps for from-scratch training:


from torch.optim.lr_scheduler import OneCycleLR
#scheduler = OneCycleLR(optimizer, max_lr=3e-3, steps_per_epoch=len(train_loader), epochs=EPOCHS)

optimizer = optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
scheduler = OneCycleLR(optimizer, max_lr=5e-3,
                       steps_per_epoch=len(train_loader),
                       epochs=EPOCHS)

scaler = torch.amp.GradScaler(enabled=False)

# TRAINGIN LOOP

print("\n Beginning Training:")
start_time = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()
    model.train()
    running_loss = 0.0
    image_count = 0
    prev_100_images_start = time.time()

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        #  Forward pass in mixed precision
        #with torch.amp.autocast(device_type="cuda"):
        outputs = model(images)
        loss = criterion(outputs, labels)

        #  Scaled backward pass
        scaler.scale(loss).backward()
        
        if not torch.isfinite(loss):
            print("Non-finite loss, stopping training")
            break

        #  Unscale + clip gradients
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)

        #  Step with scaler
        scaler.step(optimizer)
        scaler.update()

        #  Step LR scheduler once per batch
        scheduler.step()

        running_loss += loss.item()
        image_count += images.size(0)

        # Occasional progress report
        if image_count % 5000 == 0:
            print(f"Trained {image_count} images, Time: {time.time() - prev_100_images_start:.1f}s")
            prev_100_images_start = time.time()

    avg_loss = running_loss / len(train_loader)


    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()


    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Loss: {avg_loss:.4f} | Test Accuracy: {100 * correct / total:.2f}% | Time: {time.time()-epoch_start:.1f}s")


print(f"Finished Training on {image_count} images in {time.time() - start_time} seconds")

In [ ]:
# Results
 Beginning Training:
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.8s
Epoch 01/200 | Loss: 0.6849 | Test Accuracy: 56.98% | Time: 67.2s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 24.7s
Epoch 02/200 | Loss: 0.6591 | Test Accuracy: 59.13% | Time: 67.6s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.1s
Epoch 03/200 | Loss: 0.6244 | Test Accuracy: 64.39% | Time: 67.7s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.0s
Epoch 04/200 | Loss: 0.6137 | Test Accuracy: 66.32% | Time: 68.2s
Trained 10000 images, Time: 25.3s
Trained 20000 images, Time: 25.2s
Epoch 05/200 | Loss: 0.5963 | Test Accuracy: 64.79% | Time: 68.8s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.0s
Epoch 06/200 | Loss: 0.5948 | Test Accuracy: 67.27% | Time: 67.8s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.0s
Epoch 07/200 | Loss: 0.5794 | Test Accuracy: 67.76% | Time: 67.8s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.3s
Epoch 08/200 | Loss: 0.5823 | Test Accuracy: 68.33% | Time: 67.7s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.8s
Epoch 09/200 | Loss: 0.5814 | Test Accuracy: 68.67% | Time: 68.0s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.7s
Epoch 10/200 | Loss: 0.5819 | Test Accuracy: 68.72% | Time: 66.8s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 25.3s
Epoch 11/200 | Loss: 0.5748 | Test Accuracy: 65.62% | Time: 68.1s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.8s
Epoch 12/200 | Loss: 0.5704 | Test Accuracy: 70.06% | Time: 67.9s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.6s
Epoch 13/200 | Loss: 0.5578 | Test Accuracy: 65.58% | Time: 66.7s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.7s
Epoch 14/200 | Loss: 0.5681 | Test Accuracy: 69.41% | Time: 67.6s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 25.0s
Epoch 15/200 | Loss: 0.5550 | Test Accuracy: 60.28% | Time: 67.3s
Trained 10000 images, Time: 24.6s
Trained 20000 images, Time: 25.0s
Epoch 16/200 | Loss: 0.5482 | Test Accuracy: 68.41% | Time: 66.9s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 25.6s
Epoch 17/200 | Loss: 0.5413 | Test Accuracy: 68.06% | Time: 68.7s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 25.4s
Epoch 18/200 | Loss: 0.5482 | Test Accuracy: 70.16% | Time: 68.2s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.9s
Epoch 19/200 | Loss: 0.5330 | Test Accuracy: 67.68% | Time: 67.4s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.7s
Epoch 20/200 | Loss: 0.5286 | Test Accuracy: 59.81% | Time: 67.3s
Trained 10000 images, Time: 25.5s
Trained 20000 images, Time: 24.6s
Epoch 21/200 | Loss: 0.5161 | Test Accuracy: 70.36% | Time: 67.7s
Trained 10000 images, Time: 24.6s
Trained 20000 images, Time: 24.7s
Epoch 22/200 | Loss: 0.5103 | Test Accuracy: 71.70% | Time: 67.1s
Trained 10000 images, Time: 25.3s
Trained 20000 images, Time: 24.8s
Epoch 23/200 | Loss: 0.5015 | Test Accuracy: 65.08% | Time: 67.5s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.9s
Epoch 24/200 | Loss: 0.4874 | Test Accuracy: 69.51% | Time: 67.2s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.9s
Epoch 25/200 | Loss: 0.4832 | Test Accuracy: 64.41% | Time: 67.3s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.9s
Epoch 26/200 | Loss: 0.4852 | Test Accuracy: 67.78% | Time: 67.8s
Trained 10000 images, Time: 25.7s
Trained 20000 images, Time: 25.3s
Epoch 27/200 | Loss: 0.4792 | Test Accuracy: 64.18% | Time: 68.6s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 25.0s
Epoch 28/200 | Loss: 0.4702 | Test Accuracy: 65.67% | Time: 67.6s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.9s
Epoch 29/200 | Loss: 0.4630 | Test Accuracy: 73.27% | Time: 67.4s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.8s
Epoch 30/200 | Loss: 0.4563 | Test Accuracy: 71.70% | Time: 66.9s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 25.5s
Epoch 31/200 | Loss: 0.4528 | Test Accuracy: 72.02% | Time: 67.7s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.8s
Epoch 32/200 | Loss: 0.4498 | Test Accuracy: 72.22% | Time: 67.2s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.6s
Epoch 33/200 | Loss: 0.4357 | Test Accuracy: 69.94% | Time: 67.5s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.9s
Epoch 34/200 | Loss: 0.4409 | Test Accuracy: 65.36% | Time: 67.4s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.7s
Epoch 35/200 | Loss: 0.4365 | Test Accuracy: 72.70% | Time: 66.8s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 25.3s
Epoch 36/200 | Loss: 0.4258 | Test Accuracy: 70.12% | Time: 67.5s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 25.1s
Epoch 37/200 | Loss: 0.4187 | Test Accuracy: 74.01% | Time: 67.2s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 24.9s
Epoch 38/200 | Loss: 0.4170 | Test Accuracy: 72.98% | Time: 67.5s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.7s
Epoch 39/200 | Loss: 0.4109 | Test Accuracy: 72.59% | Time: 66.9s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.8s
Epoch 40/200 | Loss: 0.4036 | Test Accuracy: 66.10% | Time: 67.1s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.8s
Epoch 41/200 | Loss: 0.3950 | Test Accuracy: 71.87% | Time: 67.1s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.6s
Epoch 42/200 | Loss: 0.3937 | Test Accuracy: 69.07% | Time: 66.9s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.7s
Epoch 43/200 | Loss: 0.3883 | Test Accuracy: 68.84% | Time: 66.7s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.7s
Epoch 44/200 | Loss: 0.3803 | Test Accuracy: 71.58% | Time: 67.0s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.7s
Epoch 45/200 | Loss: 0.3798 | Test Accuracy: 68.50% | Time: 67.5s
Trained 10000 images, Time: 25.4s
Trained 20000 images, Time: 24.8s
Epoch 46/200 | Loss: 0.3763 | Test Accuracy: 70.84% | Time: 67.7s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.9s
Epoch 47/200 | Loss: 0.3665 | Test Accuracy: 73.68% | Time: 66.9s
Trained 10000 images, Time: 25.7s
Trained 20000 images, Time: 24.9s
Epoch 48/200 | Loss: 0.3651 | Test Accuracy: 71.12% | Time: 67.9s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.7s
Epoch 49/200 | Loss: 0.3550 | Test Accuracy: 74.02% | Time: 67.1s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.7s
Epoch 50/200 | Loss: 0.3562 | Test Accuracy: 74.24% | Time: 67.2s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.8s
Epoch 51/200 | Loss: 0.3457 | Test Accuracy: 72.63% | Time: 66.9s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.7s
Epoch 52/200 | Loss: 0.3384 | Test Accuracy: 76.00% | Time: 66.9s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.7s
Epoch 53/200 | Loss: 0.3324 | Test Accuracy: 72.29% | Time: 67.1s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.7s
Epoch 54/200 | Loss: 0.3226 | Test Accuracy: 71.94% | Time: 66.8s
Trained 10000 images, Time: 25.3s
Trained 20000 images, Time: 25.0s
Epoch 55/200 | Loss: 0.3218 | Test Accuracy: 71.12% | Time: 68.1s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.9s
Epoch 56/200 | Loss: 0.3154 | Test Accuracy: 73.60% | Time: 67.0s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.7s
Epoch 57/200 | Loss: 0.3036 | Test Accuracy: 74.14% | Time: 67.0s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.6s
Epoch 58/200 | Loss: 0.3023 | Test Accuracy: 72.44% | Time: 66.9s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 25.1s
Epoch 59/200 | Loss: 0.3020 | Test Accuracy: 75.68% | Time: 67.8s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.7s
Epoch 60/200 | Loss: 0.2884 | Test Accuracy: 73.40% | Time: 66.9s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.6s
Epoch 61/200 | Loss: 0.2879 | Test Accuracy: 72.58% | Time: 67.3s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.6s
Epoch 62/200 | Loss: 0.2818 | Test Accuracy: 73.71% | Time: 66.8s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.8s
Epoch 63/200 | Loss: 0.2728 | Test Accuracy: 75.44% | Time: 67.1s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 25.2s
Epoch 64/200 | Loss: 0.2721 | Test Accuracy: 73.58% | Time: 67.9s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.7s
Epoch 65/200 | Loss: 0.2627 | Test Accuracy: 73.51% | Time: 66.9s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.7s
Epoch 66/200 | Loss: 0.2536 | Test Accuracy: 73.66% | Time: 67.0s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.8s
Epoch 67/200 | Loss: 0.2518 | Test Accuracy: 74.41% | Time: 67.1s
Trained 10000 images, Time: 25.5s
Trained 20000 images, Time: 24.7s
Epoch 68/200 | Loss: 0.2478 | Test Accuracy: 74.79% | Time: 67.6s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.0s
Epoch 69/200 | Loss: 0.2398 | Test Accuracy: 76.78% | Time: 67.8s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 24.8s
Epoch 70/200 | Loss: 0.2348 | Test Accuracy: 73.68% | Time: 67.5s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.9s
Epoch 71/200 | Loss: 0.2229 | Test Accuracy: 75.62% | Time: 67.1s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.0s
Epoch 72/200 | Loss: 0.2260 | Test Accuracy: 74.22% | Time: 67.2s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 24.8s
Epoch 73/200 | Loss: 0.2209 | Test Accuracy: 75.24% | Time: 67.3s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 24.9s
Epoch 74/200 | Loss: 0.2190 | Test Accuracy: 75.81% | Time: 67.4s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.8s
Epoch 75/200 | Loss: 0.2126 | Test Accuracy: 74.63% | Time: 66.9s
Trained 10000 images, Time: 25.3s
Trained 20000 images, Time: 24.6s
Epoch 76/200 | Loss: 0.2081 | Test Accuracy: 73.02% | Time: 67.4s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.9s
Epoch 77/200 | Loss: 0.2022 | Test Accuracy: 75.50% | Time: 67.5s
Trained 10000 images, Time: 25.9s
Trained 20000 images, Time: 25.1s
Epoch 78/200 | Loss: 0.1976 | Test Accuracy: 73.06% | Time: 68.4s
Trained 10000 images, Time: 25.3s
Trained 20000 images, Time: 25.4s
Epoch 79/200 | Loss: 0.1976 | Test Accuracy: 75.47% | Time: 68.1s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.9s
Epoch 80/200 | Loss: 0.1897 | Test Accuracy: 77.33% | Time: 67.3s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.8s
Epoch 81/200 | Loss: 0.1857 | Test Accuracy: 74.31% | Time: 67.0s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 24.8s
Epoch 82/200 | Loss: 0.1849 | Test Accuracy: 76.78% | Time: 67.3s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.5s
Epoch 83/200 | Loss: 0.1780 | Test Accuracy: 76.13% | Time: 68.2s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.5s
Epoch 84/200 | Loss: 0.1729 | Test Accuracy: 72.80% | Time: 66.9s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.7s
Epoch 85/200 | Loss: 0.1672 | Test Accuracy: 75.69% | Time: 66.9s
Trained 10000 images, Time: 24.6s
Trained 20000 images, Time: 24.7s
Epoch 86/200 | Loss: 0.1685 | Test Accuracy: 77.23% | Time: 66.8s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.8s
Epoch 87/200 | Loss: 0.1658 | Test Accuracy: 78.11% | Time: 67.0s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 24.8s
Epoch 88/200 | Loss: 0.1643 | Test Accuracy: 76.14% | Time: 67.3s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.9s
Epoch 89/200 | Loss: 0.1578 | Test Accuracy: 76.39% | Time: 67.3s
Trained 10000 images, Time: 25.6s
Trained 20000 images, Time: 24.9s
Epoch 90/200 | Loss: 0.1576 | Test Accuracy: 76.54% | Time: 69.2s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 25.0s
Epoch 91/200 | Loss: 0.1515 | Test Accuracy: 75.43% | Time: 68.2s
Trained 10000 images, Time: 25.5s
Trained 20000 images, Time: 25.2s
Epoch 92/200 | Loss: 0.1538 | Test Accuracy: 73.23% | Time: 68.6s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.8s
Epoch 93/200 | Loss: 0.1447 | Test Accuracy: 75.97% | Time: 67.3s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.7s
Epoch 94/200 | Loss: 0.1375 | Test Accuracy: 77.32% | Time: 67.4s
Trained 10000 images, Time: 24.6s
Trained 20000 images, Time: 24.6s
Epoch 95/200 | Loss: 0.1419 | Test Accuracy: 77.80% | Time: 66.6s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 25.1s
Epoch 96/200 | Loss: 0.1391 | Test Accuracy: 73.89% | Time: 67.5s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.7s
Epoch 97/200 | Loss: 0.1344 | Test Accuracy: 77.21% | Time: 67.3s
Trained 10000 images, Time: 25.5s
Trained 20000 images, Time: 24.9s
Epoch 98/200 | Loss: 0.1284 | Test Accuracy: 78.19% | Time: 67.6s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 25.0s
Epoch 99/200 | Loss: 0.1251 | Test Accuracy: 77.74% | Time: 67.2s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.9s
Epoch 100/200 | Loss: 0.1260 | Test Accuracy: 76.76% | Time: 67.1s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.8s
Epoch 101/200 | Loss: 0.1228 | Test Accuracy: 75.74% | Time: 67.3s
Trained 10000 images, Time: 25.4s
Trained 20000 images, Time: 25.5s
Epoch 102/200 | Loss: 0.1194 | Test Accuracy: 74.67% | Time: 68.5s
Trained 10000 images, Time: 25.8s
Trained 20000 images, Time: 25.2s
Epoch 103/200 | Loss: 0.1183 | Test Accuracy: 74.14% | Time: 68.7s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 25.1s
Epoch 104/200 | Loss: 0.1231 | Test Accuracy: 75.39% | Time: 67.9s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.7s
Epoch 105/200 | Loss: 0.1167 | Test Accuracy: 77.43% | Time: 67.1s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.7s
Epoch 106/200 | Loss: 0.1087 | Test Accuracy: 74.50% | Time: 67.9s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.8s
Epoch 107/200 | Loss: 0.1115 | Test Accuracy: 77.97% | Time: 66.8s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 25.0s
Epoch 108/200 | Loss: 0.1087 | Test Accuracy: 77.13% | Time: 67.2s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.8s
Epoch 109/200 | Loss: 0.1050 | Test Accuracy: 76.74% | Time: 67.1s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.7s
Epoch 110/200 | Loss: 0.1010 | Test Accuracy: 76.83% | Time: 67.1s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 25.3s
Epoch 111/200 | Loss: 0.1004 | Test Accuracy: 77.77% | Time: 68.1s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 24.9s
Epoch 112/200 | Loss: 0.0980 | Test Accuracy: 75.32% | Time: 67.6s
Trained 10000 images, Time: 25.3s
Trained 20000 images, Time: 24.7s
Epoch 113/200 | Loss: 0.0937 | Test Accuracy: 76.48% | Time: 67.5s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 25.0s
Epoch 114/200 | Loss: 0.0941 | Test Accuracy: 75.41% | Time: 68.0s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 25.4s
Epoch 115/200 | Loss: 0.0923 | Test Accuracy: 76.43% | Time: 69.1s
Trained 10000 images, Time: 25.3s
Trained 20000 images, Time: 25.1s
Epoch 116/200 | Loss: 0.0850 | Test Accuracy: 75.92% | Time: 67.9s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.4s
Epoch 117/200 | Loss: 0.0888 | Test Accuracy: 78.08% | Time: 68.3s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.3s
Epoch 118/200 | Loss: 0.0920 | Test Accuracy: 77.28% | Time: 68.0s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 25.1s
Epoch 119/200 | Loss: 0.0836 | Test Accuracy: 75.10% | Time: 67.6s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.0s
Epoch 120/200 | Loss: 0.0796 | Test Accuracy: 77.96% | Time: 68.0s
Trained 10000 images, Time: 25.4s
Trained 20000 images, Time: 25.2s
Epoch 121/200 | Loss: 0.0821 | Test Accuracy: 74.54% | Time: 68.0s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.1s
Epoch 122/200 | Loss: 0.0720 | Test Accuracy: 77.38% | Time: 67.8s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 24.6s
Epoch 123/200 | Loss: 0.0724 | Test Accuracy: 75.54% | Time: 67.8s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 24.8s
Epoch 124/200 | Loss: 0.0763 | Test Accuracy: 77.10% | Time: 67.4s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.8s
Epoch 125/200 | Loss: 0.0702 | Test Accuracy: 77.97% | Time: 67.1s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 24.7s
Epoch 126/200 | Loss: 0.0705 | Test Accuracy: 78.01% | Time: 67.5s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.7s
Epoch 127/200 | Loss: 0.0700 | Test Accuracy: 75.14% | Time: 69.2s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.9s
Epoch 128/200 | Loss: 0.0708 | Test Accuracy: 76.60% | Time: 67.4s
Trained 10000 images, Time: 25.4s
Trained 20000 images, Time: 25.0s
Epoch 129/200 | Loss: 0.0665 | Test Accuracy: 77.86% | Time: 68.2s
Trained 10000 images, Time: 25.4s
Trained 20000 images, Time: 25.4s
Epoch 130/200 | Loss: 0.0611 | Test Accuracy: 77.31% | Time: 68.3s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.9s
Epoch 131/200 | Loss: 0.0641 | Test Accuracy: 78.14% | Time: 67.1s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.9s
Epoch 132/200 | Loss: 0.0604 | Test Accuracy: 77.41% | Time: 67.6s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.9s
Epoch 133/200 | Loss: 0.0612 | Test Accuracy: 77.40% | Time: 67.0s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.7s
Epoch 134/200 | Loss: 0.0622 | Test Accuracy: 77.13% | Time: 67.0s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.1s
Epoch 135/200 | Loss: 0.0567 | Test Accuracy: 77.78% | Time: 67.6s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 25.0s
Epoch 136/200 | Loss: 0.0551 | Test Accuracy: 76.96% | Time: 67.6s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.9s
Epoch 137/200 | Loss: 0.0534 | Test Accuracy: 75.98% | Time: 67.2s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 25.2s
Epoch 138/200 | Loss: 0.0525 | Test Accuracy: 75.58% | Time: 67.3s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.1s
Epoch 139/200 | Loss: 0.0513 | Test Accuracy: 76.71% | Time: 67.9s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 24.9s
Epoch 140/200 | Loss: 0.0461 | Test Accuracy: 77.02% | Time: 67.6s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 25.2s
Epoch 141/200 | Loss: 0.0494 | Test Accuracy: 77.33% | Time: 67.4s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.0s
Epoch 142/200 | Loss: 0.0447 | Test Accuracy: 76.60% | Time: 67.7s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 25.0s
Epoch 143/200 | Loss: 0.0430 | Test Accuracy: 77.47% | Time: 67.3s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.7s
Epoch 144/200 | Loss: 0.0424 | Test Accuracy: 77.64% | Time: 67.1s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 25.0s
Epoch 145/200 | Loss: 0.0443 | Test Accuracy: 78.11% | Time: 67.2s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.1s
Epoch 146/200 | Loss: 0.0427 | Test Accuracy: 76.01% | Time: 68.0s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.9s
Epoch 147/200 | Loss: 0.0400 | Test Accuracy: 78.51% | Time: 67.6s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 25.1s
Epoch 148/200 | Loss: 0.0390 | Test Accuracy: 76.91% | Time: 68.2s
Trained 10000 images, Time: 25.9s
Trained 20000 images, Time: 24.8s
Epoch 149/200 | Loss: 0.0379 | Test Accuracy: 78.28% | Time: 68.1s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.6s
Epoch 150/200 | Loss: 0.0362 | Test Accuracy: 76.77% | Time: 66.8s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.8s
Epoch 151/200 | Loss: 0.0362 | Test Accuracy: 76.43% | Time: 66.9s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 25.0s
Epoch 152/200 | Loss: 0.0368 | Test Accuracy: 76.89% | Time: 67.7s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 25.1s
Epoch 153/200 | Loss: 0.0329 | Test Accuracy: 77.79% | Time: 67.9s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.9s
Epoch 154/200 | Loss: 0.0357 | Test Accuracy: 76.80% | Time: 67.5s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 25.1s
Epoch 155/200 | Loss: 0.0330 | Test Accuracy: 76.98% | Time: 67.7s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.1s
Epoch 156/200 | Loss: 0.0310 | Test Accuracy: 78.28% | Time: 67.6s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.0s
Epoch 157/200 | Loss: 0.0313 | Test Accuracy: 77.71% | Time: 67.7s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 25.4s
Epoch 158/200 | Loss: 0.0270 | Test Accuracy: 76.88% | Time: 67.8s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.9s
Epoch 159/200 | Loss: 0.0294 | Test Accuracy: 76.70% | Time: 67.2s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 24.8s
Epoch 160/200 | Loss: 0.0279 | Test Accuracy: 77.53% | Time: 67.3s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 24.9s
Epoch 161/200 | Loss: 0.0271 | Test Accuracy: 76.32% | Time: 67.6s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 24.9s
Epoch 162/200 | Loss: 0.0257 | Test Accuracy: 76.92% | Time: 67.7s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.8s
Epoch 163/200 | Loss: 0.0242 | Test Accuracy: 76.61% | Time: 67.3s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.7s
Epoch 164/200 | Loss: 0.0247 | Test Accuracy: 77.24% | Time: 66.9s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.6s
Epoch 165/200 | Loss: 0.0214 | Test Accuracy: 78.00% | Time: 67.0s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.3s
Epoch 166/200 | Loss: 0.0209 | Test Accuracy: 77.00% | Time: 67.9s
Trained 10000 images, Time: 26.2s
Trained 20000 images, Time: 25.2s
Epoch 167/200 | Loss: 0.0208 | Test Accuracy: 77.40% | Time: 69.5s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 24.9s
Epoch 168/200 | Loss: 0.0220 | Test Accuracy: 77.31% | Time: 67.7s
Trained 10000 images, Time: 25.3s
Trained 20000 images, Time: 25.3s
Epoch 169/200 | Loss: 0.0212 | Test Accuracy: 77.19% | Time: 68.0s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.8s
Epoch 170/200 | Loss: 0.0217 | Test Accuracy: 76.91% | Time: 67.6s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.1s
Epoch 171/200 | Loss: 0.0203 | Test Accuracy: 77.03% | Time: 67.6s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.1s
Epoch 172/200 | Loss: 0.0180 | Test Accuracy: 78.49% | Time: 67.7s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 24.9s
Epoch 173/200 | Loss: 0.0174 | Test Accuracy: 77.32% | Time: 67.5s
Trained 10000 images, Time: 25.3s
Trained 20000 images, Time: 24.7s
Epoch 174/200 | Loss: 0.0178 | Test Accuracy: 77.80% | Time: 67.4s
Trained 10000 images, Time: 25.4s
Trained 20000 images, Time: 24.7s
Epoch 175/200 | Loss: 0.0161 | Test Accuracy: 76.81% | Time: 67.5s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 25.0s
Epoch 176/200 | Loss: 0.0167 | Test Accuracy: 77.57% | Time: 67.9s
Trained 10000 images, Time: 26.0s
Trained 20000 images, Time: 25.6s
Epoch 177/200 | Loss: 0.0138 | Test Accuracy: 77.29% | Time: 69.2s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 24.8s
Epoch 178/200 | Loss: 0.0148 | Test Accuracy: 76.64% | Time: 67.4s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.9s
Epoch 179/200 | Loss: 0.0153 | Test Accuracy: 77.56% | Time: 67.8s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.9s
Epoch 180/200 | Loss: 0.0149 | Test Accuracy: 77.11% | Time: 67.8s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.0s
Epoch 181/200 | Loss: 0.0146 | Test Accuracy: 77.56% | Time: 67.9s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.1s
Epoch 182/200 | Loss: 0.0130 | Test Accuracy: 77.73% | Time: 67.5s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.8s
Epoch 183/200 | Loss: 0.0126 | Test Accuracy: 77.01% | Time: 67.8s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 25.1s
Epoch 184/200 | Loss: 0.0133 | Test Accuracy: 77.78% | Time: 67.9s
Trained 10000 images, Time: 25.3s
Trained 20000 images, Time: 24.8s
Epoch 185/200 | Loss: 0.0119 | Test Accuracy: 77.18% | Time: 68.2s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 25.4s
Epoch 186/200 | Loss: 0.0141 | Test Accuracy: 77.70% | Time: 68.3s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.8s
Epoch 187/200 | Loss: 0.0106 | Test Accuracy: 77.27% | Time: 67.5s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.3s
Epoch 188/200 | Loss: 0.0120 | Test Accuracy: 77.57% | Time: 67.7s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.2s
Epoch 189/200 | Loss: 0.0135 | Test Accuracy: 77.18% | Time: 67.8s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.9s
Epoch 190/200 | Loss: 0.0104 | Test Accuracy: 77.43% | Time: 67.5s
Trained 10000 images, Time: 25.4s
Trained 20000 images, Time: 25.2s
Epoch 191/200 | Loss: 0.0098 | Test Accuracy: 77.47% | Time: 68.2s
Trained 10000 images, Time: 25.1s

Version: (2,2,6,2), one cycle lr (broken) peak 76.31% @ 41

In [ ]:
Train length: 21520, Test length: 9000
tensor(-0.0415) tensor(0.9638)
Using device: cuda
Total parameters: 4,778,258

Beginning Training:
  Trained 10000 images, Time: 38.2s
  Trained 20000 images, Time: 36.7s
  *** New best accuracy: 55.10% ***
Epoch 001/300 | Loss: 0.6890 | Test Acc: 55.10% | LR: 0.000082 | Time: 97.5s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 36.4s
  *** New best accuracy: 60.46% ***
Epoch 002/300 | Loss: 0.6626 | Test Acc: 60.46% | LR: 0.000089 | Time: 95.7s
  Trained 10000 images, Time: 38.6s
  Trained 20000 images, Time: 36.8s
  *** New best accuracy: 66.14% ***
Epoch 003/300 | Loss: 0.6224 | Test Acc: 66.14% | LR: 0.000101 | Time: 98.5s
  Trained 10000 images, Time: 37.7s
  Trained 20000 images, Time: 38.3s
  *** New best accuracy: 67.27% ***
Epoch 004/300 | Loss: 0.6072 | Test Acc: 67.27% | LR: 0.000117 | Time: 98.5s
  Trained 10000 images, Time: 37.9s
  Trained 20000 images, Time: 37.8s
Epoch 005/300 | Loss: 0.6021 | Test Acc: 54.66% | LR: 0.000138 | Time: 98.9s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 37.6s
  *** New best accuracy: 68.92% ***
Epoch 006/300 | Loss: 0.5916 | Test Acc: 68.92% | LR: 0.000163 | Time: 98.5s
  Trained 10000 images, Time: 37.2s
  Trained 20000 images, Time: 37.7s
  *** New best accuracy: 69.10% ***
Epoch 007/300 | Loss: 0.5904 | Test Acc: 69.10% | LR: 0.000192 | Time: 98.4s
  Trained 10000 images, Time: 37.2s
  Trained 20000 images, Time: 37.0s
  *** New best accuracy: 69.80% ***
Epoch 008/300 | Loss: 0.5852 | Test Acc: 69.80% | LR: 0.000226 | Time: 96.5s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 37.3s
Epoch 009/300 | Loss: 0.5718 | Test Acc: 67.04% | LR: 0.000263 | Time: 96.7s
  Trained 10000 images, Time: 37.1s
  Trained 20000 images, Time: 37.2s
Epoch 010/300 | Loss: 0.5620 | Test Acc: 69.09% | LR: 0.000305 | Time: 97.8s
  Trained 10000 images, Time: 37.4s
  Trained 20000 images, Time: 38.3s
Epoch 011/300 | Loss: 0.5568 | Test Acc: 62.83% | LR: 0.000349 | Time: 98.0s
  Trained 10000 images, Time: 38.2s
  Trained 20000 images, Time: 40.6s
Epoch 012/300 | Loss: 0.5409 | Test Acc: 69.78% | LR: 0.000398 | Time: 102.8s
  Trained 10000 images, Time: 40.5s
  Trained 20000 images, Time: 36.7s
Epoch 013/300 | Loss: 0.5377 | Test Acc: 68.30% | LR: 0.000449 | Time: 100.0s
  Trained 10000 images, Time: 37.3s
  Trained 20000 images, Time: 37.1s
Epoch 014/300 | Loss: 0.5116 | Test Acc: 68.72% | LR: 0.000503 | Time: 96.6s
  Trained 10000 images, Time: 37.1s
  Trained 20000 images, Time: 36.9s
  *** New best accuracy: 72.46% ***
Epoch 015/300 | Loss: 0.5082 | Test Acc: 72.46% | LR: 0.000560 | Time: 96.7s
  Trained 10000 images, Time: 36.5s
  Trained 20000 images, Time: 36.8s
Epoch 016/300 | Loss: 0.5026 | Test Acc: 64.04% | LR: 0.000619 | Time: 95.8s
  Trained 10000 images, Time: 37.1s
  Trained 20000 images, Time: 38.2s
Epoch 017/300 | Loss: 0.4892 | Test Acc: 71.61% | LR: 0.000680 | Time: 98.3s
  Trained 10000 images, Time: 36.4s
  Trained 20000 images, Time: 37.4s
Epoch 018/300 | Loss: 0.4806 | Test Acc: 65.77% | LR: 0.000743 | Time: 97.6s
  Trained 10000 images, Time: 36.9s
  Trained 20000 images, Time: 37.2s
Epoch 019/300 | Loss: 0.4705 | Test Acc: 72.42% | LR: 0.000808 | Time: 96.9s
  Trained 10000 images, Time: 37.1s
  Trained 20000 images, Time: 37.4s
Epoch 020/300 | Loss: 0.4607 | Test Acc: 71.49% | LR: 0.000873 | Time: 97.4s
  Trained 10000 images, Time: 37.3s
  Trained 20000 images, Time: 37.1s
  *** New best accuracy: 74.09% ***
Epoch 021/300 | Loss: 0.4577 | Test Acc: 74.09% | LR: 0.000940 | Time: 96.0s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.9s
  *** New best accuracy: 74.39% ***
Epoch 022/300 | Loss: 0.4438 | Test Acc: 74.39% | LR: 0.001007 | Time: 95.8s
  Trained 10000 images, Time: 36.2s
  Trained 20000 images, Time: 36.6s
  *** New best accuracy: 74.48% ***
Epoch 023/300 | Loss: 0.4358 | Test Acc: 74.48% | LR: 0.001074 | Time: 95.1s
  Trained 10000 images, Time: 36.9s
  Trained 20000 images, Time: 37.5s
  *** New best accuracy: 75.03% ***
Epoch 024/300 | Loss: 0.4291 | Test Acc: 75.03% | LR: 0.001140 | Time: 97.0s
  Trained 10000 images, Time: 36.2s
  Trained 20000 images, Time: 37.0s
Epoch 025/300 | Loss: 0.4173 | Test Acc: 71.20% | LR: 0.001207 | Time: 95.9s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 36.2s
Epoch 026/300 | Loss: 0.4091 | Test Acc: 71.91% | LR: 0.001272 | Time: 94.6s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 37.0s
Epoch 027/300 | Loss: 0.3975 | Test Acc: 71.06% | LR: 0.001337 | Time: 96.7s
  Trained 10000 images, Time: 36.9s
  Trained 20000 images, Time: 36.5s
Epoch 028/300 | Loss: 0.3834 | Test Acc: 72.17% | LR: 0.001400 | Time: 95.0s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 36.6s
Epoch 029/300 | Loss: 0.3820 | Test Acc: 73.31% | LR: 0.001461 | Time: 95.9s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 36.9s
Epoch 030/300 | Loss: 0.3687 | Test Acc: 72.59% | LR: 0.001520 | Time: 95.5s
  Trained 10000 images, Time: 38.3s
  Trained 20000 images, Time: 37.9s
Epoch 031/300 | Loss: 0.3616 | Test Acc: 74.31% | LR: 0.001577 | Time: 99.0s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 36.5s
Epoch 032/300 | Loss: 0.3527 | Test Acc: 74.62% | LR: 0.001631 | Time: 96.0s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.8s
Epoch 033/300 | Loss: 0.3423 | Test Acc: 73.61% | LR: 0.001682 | Time: 95.6s
  Trained 10000 images, Time: 37.0s
  Trained 20000 images, Time: 36.6s
Epoch 034/300 | Loss: 0.3333 | Test Acc: 74.50% | LR: 0.001731 | Time: 96.5s
  Trained 10000 images, Time: 36.5s
  Trained 20000 images, Time: 36.7s
Epoch 035/300 | Loss: 0.3216 | Test Acc: 74.36% | LR: 0.001775 | Time: 95.0s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 36.9s
Epoch 036/300 | Loss: 0.3111 | Test Acc: 74.21% | LR: 0.001817 | Time: 96.0s
  Trained 10000 images, Time: 37.2s
  Trained 20000 images, Time: 37.6s
  *** New best accuracy: 75.24% ***
Epoch 037/300 | Loss: 0.3025 | Test Acc: 75.24% | LR: 0.001854 | Time: 96.1s
  Trained 10000 images, Time: 37.2s
  Trained 20000 images, Time: 37.1s
Epoch 038/300 | Loss: 0.2947 | Test Acc: 75.16% | LR: 0.001888 | Time: 96.9s
  Trained 10000 images, Time: 36.2s
  Trained 20000 images, Time: 36.7s
  *** New best accuracy: 75.87% ***
Epoch 039/300 | Loss: 0.2846 | Test Acc: 75.87% | LR: 0.001917 | Time: 95.4s
  Trained 10000 images, Time: 37.7s
  Trained 20000 images, Time: 38.0s
Epoch 040/300 | Loss: 0.2776 | Test Acc: 73.93% | LR: 0.001942 | Time: 99.3s
  Trained 10000 images, Time: 37.6s
  Trained 20000 images, Time: 37.2s
Epoch 041/300 | Loss: 0.2694 | Test Acc: 72.11% | LR: 0.001963 | Time: 97.6s
  Trained 10000 images, Time: 37.2s
  Trained 20000 images, Time: 37.3s
  *** New best accuracy: 76.31% ***
Epoch 042/300 | Loss: 0.2626 | Test Acc: 76.31% | LR: 0.001979 | Time: 97.2s
  Trained 10000 images, Time: 37.4s
  Trained 20000 images, Time: 37.0s
Epoch 043/300 | Loss: 0.2460 | Test Acc: 68.33% | LR: 0.001991 | Time: 97.8s
  Trained 10000 images, Time: 37.5s
  Trained 20000 images, Time: 37.9s
Epoch 044/300 | Loss: 0.2380 | Test Acc: 57.49% | LR: 0.001998 | Time: 98.1s
  Trained 10000 images, Time: 37.6s
  Trained 20000 images, Time: 37.4s
Epoch 045/300 | Loss: 0.2385 | Test Acc: 53.01% | LR: 0.002000 | Time: 97.9s
  Trained 10000 images, Time: 37.0s
  Trained 20000 images, Time: 36.7s
Epoch 046/300 | Loss: 0.2286 | Test Acc: 50.28% | LR: 0.002000 | Time: 96.4s
  Trained 10000 images, Time: 36.5s
  Trained 20000 images, Time: 37.1s
Epoch 047/300 | Loss: 0.2146 | Test Acc: 49.52% | LR: 0.002000 | Time: 95.9s
  Trained 10000 images, Time: 36.4s
  Trained 20000 images, Time: 36.6s
Epoch 048/300 | Loss: 0.2138 | Test Acc: 49.56% | LR: 0.001999 | Time: 95.7s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.4s
Epoch 049/300 | Loss: 0.2038 | Test Acc: 49.56% | LR: 0.001999 | Time: 94.8s
  Trained 10000 images, Time: 36.9s
  Trained 20000 images, Time: 37.5s
Epoch 050/300 | Loss: 0.1975 | Test Acc: 49.56% | LR: 0.001998 | Time: 97.2s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.9s
Epoch 051/300 | Loss: 0.1928 | Test Acc: 49.56% | LR: 0.001997 | Time: 95.4s
  Trained 10000 images, Time: 36.5s
  Trained 20000 images, Time: 36.9s
Epoch 052/300 | Loss: 0.1875 | Test Acc: 49.56% | LR: 0.001996 | Time: 95.7s
  Trained 10000 images, Time: 36.1s
  Trained 20000 images, Time: 37.1s
Epoch 053/300 | Loss: 0.1802 | Test Acc: 49.56% | LR: 0.001995 | Time: 96.0s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.9s
Epoch 054/300 | Loss: 0.1734 | Test Acc: 49.56% | LR: 0.001994 | Time: 95.5s
  Trained 10000 images, Time: 36.4s
  Trained 20000 images, Time: 36.9s
Epoch 055/300 | Loss: 0.1740 | Test Acc: 49.56% | LR: 0.001992 | Time: 95.9s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 36.7s
Epoch 056/300 | Loss: 0.1582 | Test Acc: 49.56% | LR: 0.001991 | Time: 95.5s
  Trained 10000 images, Time: 37.2s
  Trained 20000 images, Time: 36.9s
Epoch 057/300 | Loss: 0.1595 | Test Acc: 49.56% | LR: 0.001989 | Time: 96.9s
  Trained 10000 images, Time: 37.1s
  Trained 20000 images, Time: 37.1s
Epoch 058/300 | Loss: 0.1581 | Test Acc: 49.56% | LR: 0.001987 | Time: 95.9s
  Trained 10000 images, Time: 37.2s
  Trained 20000 images, Time: 36.8s
Epoch 059/300 | Loss: 0.1506 | Test Acc: 49.56% | LR: 0.001985 | Time: 96.2s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 37.1s
Epoch 060/300 | Loss: 0.1480 | Test Acc: 49.56% | LR: 0.001983 | Time: 95.9s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 36.7s
Epoch 061/300 | Loss: 0.1418 | Test Acc: 49.56% | LR: 0.001981 | Time: 95.9s
  Trained 10000 images, Time: 36.4s
  Trained 20000 images, Time: 36.8s
Epoch 062/300 | Loss: 0.1385 | Test Acc: 49.56% | LR: 0.001978 | Time: 96.1s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 37.1s
Epoch 063/300 | Loss: 0.1291 | Test Acc: 49.56% | LR: 0.001976 | Time: 95.9s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 36.7s
Epoch 064/300 | Loss: 0.1309 | Test Acc: 49.56% | LR: 0.001973 | Time: 96.1s
  Trained 10000 images, Time: 36.4s
  Trained 20000 images, Time: 36.4s
Epoch 065/300 | Loss: 0.1288 | Test Acc: 49.56% | LR: 0.001970 | Time: 94.5s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 36.9s
Epoch 066/300 | Loss: 0.1218 | Test Acc: 49.56% | LR: 0.001967 | Time: 96.5s
  Trained 10000 images, Time: 36.9s
  Trained 20000 images, Time: 36.9s
Epoch 067/300 | Loss: 0.1178 | Test Acc: 49.56% | LR: 0.001963 | Time: 95.1s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 36.8s
Epoch 068/300 | Loss: 0.1218 | Test Acc: 49.56% | LR: 0.001960 | Time: 95.9s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.7s
Epoch 069/300 | Loss: 0.1187 | Test Acc: 49.56% | LR: 0.001957 | Time: 95.4s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 36.8s
Epoch 070/300 | Loss: 0.1167 | Test Acc: 49.56% | LR: 0.001953 | Time: 95.9s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.9s
Epoch 071/300 | Loss: 0.1103 | Test Acc: 49.56% | LR: 0.001949 | Time: 95.8s
  Trained 10000 images, Time: 36.0s
  Trained 20000 images, Time: 36.7s
Epoch 072/300 | Loss: 0.1046 | Test Acc: 49.56% | LR: 0.001945 | Time: 94.4s
  Trained 10000 images, Time: 37.0s
  Trained 20000 images, Time: 36.6s
Epoch 073/300 | Loss: 0.1041 | Test Acc: 49.56% | LR: 0.001941 | Time: 96.1s
  Trained 10000 images, Time: 36.9s
  Trained 20000 images, Time: 36.8s
Epoch 074/300 | Loss: 0.1035 | Test Acc: 49.56% | LR: 0.001937 | Time: 95.4s
  Trained 10000 images, Time: 37.0s
  Trained 20000 images, Time: 37.0s
Epoch 075/300 | Loss: 0.1017 | Test Acc: 49.56% | LR: 0.001932 | Time: 96.8s
  Trained 10000 images, Time: 37.3s
  Trained 20000 images, Time: 37.5s
Epoch 076/300 | Loss: 0.0993 | Test Acc: 49.56% | LR: 0.001928 | Time: 96.4s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.8s
Epoch 077/300 | Loss: 0.0929 | Test Acc: 49.56% | LR: 0.001923 | Time: 95.8s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.5s
Epoch 078/300 | Loss: 0.0984 | Test Acc: 49.56% | LR: 0.001918 | Time: 95.3s
  Trained 10000 images, Time: 36.5s
  Trained 20000 images, Time: 37.1s
Epoch 079/300 | Loss: 0.0926 | Test Acc: 49.56% | LR: 0.001914 | Time: 96.5s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.9s
Epoch 080/300 | Loss: 0.0882 | Test Acc: 49.56% | LR: 0.001908 | Time: 95.6s
  Trained 10000 images, Time: 36.4s
  Trained 20000 images, Time: 36.7s
Epoch 081/300 | Loss: 0.0867 | Test Acc: 49.56% | LR: 0.001903 | Time: 94.9s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.3s
Epoch 082/300 | Loss: 0.0856 | Test Acc: 49.56% | LR: 0.001898 | Time: 95.9s
  Trained 10000 images, Time: 36.9s
  Trained 20000 images, Time: 37.4s
Epoch 083/300 | Loss: 0.0832 | Test Acc: 49.56% | LR: 0.001892 | Time: 95.8s
  Trained 10000 images, Time: 36.5s
  Trained 20000 images, Time: 36.8s
Epoch 084/300 | Loss: 0.0801 | Test Acc: 49.56% | LR: 0.001887 | Time: 95.7s
  Trained 10000 images, Time: 36.2s
  Trained 20000 images, Time: 36.5s
Epoch 085/300 | Loss: 0.0795 | Test Acc: 49.56% | LR: 0.001881 | Time: 94.4s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 37.5s
Epoch 086/300 | Loss: 0.0771 | Test Acc: 49.56% | LR: 0.001875 | Time: 96.7s
  Trained 10000 images, Time: 36.2s
  Trained 20000 images, Time: 36.4s
Epoch 087/300 | Loss: 0.0788 | Test Acc: 49.56% | LR: 0.001869 | Time: 95.0s
  Trained 10000 images, Time: 36.4s
  Trained 20000 images, Time: 36.3s
Epoch 088/300 | Loss: 0.0750 | Test Acc: 49.56% | LR: 0.001863 | Time: 94.9s
  Trained 10000 images, Time: 36.2s
  Trained 20000 images, Time: 37.0s
Epoch 089/300 | Loss: 0.0744 | Test Acc: 49.56% | LR: 0.001857 | Time: 96.3s
  Trained 10000 images, Time: 37.0s
  Trained 20000 images, Time: 36.9s
Epoch 090/300 | Loss: 0.0744 | Test Acc: 49.56% | LR: 0.001850 | Time: 95.5s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.5s
Epoch 091/300 | Loss: 0.0673 | Test Acc: 49.56% | LR: 0.001844 | Time: 95.6s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 37.0s
Epoch 092/300 | Loss: 0.0709 | Test Acc: 49.56% | LR: 0.001837 | Time: 95.1s
  Trained 10000 images, Time: 37.5s
  Trained 20000 images, Time: 37.2s
Epoch 093/300 | Loss: 0.0700 | Test Acc: 49.56% | LR: 0.001830 | Time: 98.5s
  Trained 10000 images, Time: 37.3s
  Trained 20000 images, Time: 38.2s
Epoch 094/300 | Loss: 0.0615 | Test Acc: 49.56% | LR: 0.001823 | Time: 97.5s
  Trained 10000 images, Time: 36.9s
  Trained 20000 images, Time: 36.8s
Epoch 095/300 | Loss: 0.0660 | Test Acc: 49.56% | LR: 0.001816 | Time: 96.4s
  Trained 10000 images, Time: 36.0s
  Trained 20000 images, Time: 37.1s
Epoch 096/300 | Loss: 0.0679 | Test Acc: 49.56% | LR: 0.001809 | Time: 96.2s
  Trained 10000 images, Time: 37.0s
  Trained 20000 images, Time: 36.6s
Epoch 097/300 | Loss: 0.0613 | Test Acc: 49.56% | LR: 0.001802 | Time: 95.8s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.8s
Epoch 098/300 | Loss: 0.0642 | Test Acc: 49.56% | LR: 0.001794 | Time: 96.1s
  Trained 10000 images, Time: 36.4s
  Trained 20000 images, Time: 37.4s
Epoch 099/300 | Loss: 0.0623 | Test Acc: 49.56% | LR: 0.001787 | Time: 95.4s
  Trained 10000 images, Time: 37.1s
  Trained 20000 images, Time: 37.4s
Epoch 100/300 | Loss: 0.0627 | Test Acc: 49.56% | LR: 0.001779 | Time: 97.2s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.6s
Epoch 101/300 | Loss: 0.0597 | Test Acc: 49.56% | LR: 0.001771 | Time: 94.9s
  Trained 10000 images, Time: 36.4s
  Trained 20000 images, Time: 36.7s
Epoch 102/300 | Loss: 0.0585 | Test Acc: 49.56% | LR: 0.001763 | Time: 95.3s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.7s
Epoch 103/300 | Loss: 0.0594 | Test Acc: 49.56% | LR: 0.001755 | Time: 95.6s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 37.2s
Epoch 104/300 | Loss: 0.0552 | Test Acc: 49.56% | LR: 0.001747 | Time: 95.9s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 36.7s
Epoch 105/300 | Loss: 0.0583 | Test Acc: 49.56% | LR: 0.001739 | Time: 96.2s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.8s
Epoch 106/300 | Loss: 0.0550 | Test Acc: 49.56% | LR: 0.001731 | Time: 95.2s
  Trained 10000 images, Time: 38.0s
  Trained 20000 images, Time: 37.5s
Epoch 107/300 | Loss: 0.0551 | Test Acc: 49.56% | LR: 0.001722 | Time: 97.9s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.5s
Epoch 108/300 | Loss: 0.0532 | Test Acc: 49.56% | LR: 0.001714 | Time: 94.5s
  Trained 10000 images, Time: 37.0s
  Trained 20000 images, Time: 37.0s
Epoch 109/300 | Loss: 0.0473 | Test Acc: 49.56% | LR: 0.001705 | Time: 96.2s
  Trained 10000 images, Time: 36.2s
  Trained 20000 images, Time: 36.7s
Epoch 110/300 | Loss: 0.0534 | Test Acc: 49.56% | LR: 0.001696 | Time: 95.2s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 36.4s
Epoch 111/300 | Loss: 0.0476 | Test Acc: 49.56% | LR: 0.001687 | Time: 95.7s
  Trained 10000 images, Time: 36.0s
  Trained 20000 images, Time: 36.5s
Epoch 112/300 | Loss: 0.0497 | Test Acc: 49.56% | LR: 0.001678 | Time: 94.8s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.6s
Epoch 113/300 | Loss: 0.0482 | Test Acc: 49.56% | LR: 0.001669 | Time: 95.4s
  Trained 10000 images, Time: 37.1s
  Trained 20000 images, Time: 36.6s
Epoch 114/300 | Loss: 0.0456 | Test Acc: 49.56% | LR: 0.001660 | Time: 96.7s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.7s
Epoch 115/300 | Loss: 0.0470 | Test Acc: 49.56% | LR: 0.001651 | Time: 94.7s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.6s
Epoch 116/300 | Loss: 0.0450 | Test Acc: 49.56% | LR: 0.001641 | Time: 96.2s
  Trained 10000 images, Time: 37.1s
  Trained 20000 images, Time: 37.0s
Epoch 117/300 | Loss: 0.0458 | Test Acc: 49.56% | LR: 0.001632 | Time: 95.1s
  Trained 10000 images, Time: 36.9s
  Trained 20000 images, Time: 37.0s
Epoch 118/300 | Loss: 0.0424 | Test Acc: 49.56% | LR: 0.001622 | Time: 96.6s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.6s
Epoch 119/300 | Loss: 0.0458 | Test Acc: 49.56% | LR: 0.001612 | Time: 95.1s
  Trained 10000 images, Time: 37.0s
  Trained 20000 images, Time: 37.0s
Epoch 120/300 | Loss: 0.0424 | Test Acc: 49.56% | LR: 0.001603 | Time: 96.3s
  Trained 10000 images, Time: 36.1s
  Trained 20000 images, Time: 36.6s
Epoch 121/300 | Loss: 0.0419 | Test Acc: 49.56% | LR: 0.001593 | Time: 95.1s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.4s
Epoch 122/300 | Loss: 0.0429 | Test Acc: 49.56% | LR: 0.001583 | Time: 94.9s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 37.3s
Epoch 123/300 | Loss: 0.0421 | Test Acc: 49.56% | LR: 0.001573 | Time: 96.8s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.5s
Epoch 124/300 | Loss: 0.0416 | Test Acc: 49.56% | LR: 0.001563 | Time: 94.4s
  Trained 10000 images, Time: 36.5s
  Trained 20000 images, Time: 36.5s
Epoch 125/300 | Loss: 0.0412 | Test Acc: 49.56% | LR: 0.001552 | Time: 95.3s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 37.3s
Epoch 126/300 | Loss: 0.0385 | Test Acc: 49.56% | LR: 0.001542 | Time: 95.5s
  Trained 10000 images, Time: 37.0s
  Trained 20000 images, Time: 37.8s
Epoch 127/300 | Loss: 0.0371 | Test Acc: 49.56% | LR: 0.001532 | Time: 98.1s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 36.8s
Epoch 128/300 | Loss: 0.0413 | Test Acc: 49.56% | LR: 0.001521 | Time: 95.9s
  Trained 10000 images, Time: 36.5s
  Trained 20000 images, Time: 36.5s
Epoch 129/300 | Loss: 0.0365 | Test Acc: 49.56% | LR: 0.001511 | Time: 96.2s
  Trained 10000 images, Time: 36.9s
  Trained 20000 images, Time: 36.9s
Epoch 130/300 | Loss: 0.0382 | Test Acc: 49.56% | LR: 0.001500 | Time: 96.5s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.2s
Epoch 131/300 | Loss: 0.0351 | Test Acc: 49.56% | LR: 0.001489 | Time: 94.2s
  Trained 10000 images, Time: 36.4s
  Trained 20000 images, Time: 36.9s
Epoch 132/300 | Loss: 0.0367 | Test Acc: 49.56% | LR: 0.001479 | Time: 95.8s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 37.4s
Epoch 133/300 | Loss: 0.0333 | Test Acc: 49.56% | LR: 0.001468 | Time: 95.7s
  Trained 10000 images, Time: 37.1s
  Trained 20000 images, Time: 36.7s
Epoch 134/300 | Loss: 0.0343 | Test Acc: 49.56% | LR: 0.001457 | Time: 96.2s
  Trained 10000 images, Time: 36.4s
  Trained 20000 images, Time: 36.4s
Epoch 135/300 | Loss: 0.0349 | Test Acc: 49.56% | LR: 0.001446 | Time: 94.5s
  Trained 10000 images, Time: 36.5s
  Trained 20000 images, Time: 36.7s
Epoch 136/300 | Loss: 0.0335 | Test Acc: 49.56% | LR: 0.001435 | Time: 95.9s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 37.1s
Epoch 137/300 | Loss: 0.0330 | Test Acc: 49.56% | LR: 0.001424 | Time: 96.5s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.7s
Epoch 138/300 | Loss: 0.0332 | Test Acc: 49.56% | LR: 0.001412 | Time: 95.6s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.7s
Epoch 139/300 | Loss: 0.0306 | Test Acc: 49.56% | LR: 0.001401 | Time: 95.1s
  Trained 10000 images, Time: 36.9s
  Trained 20000 images, Time: 36.7s
Epoch 140/300 | Loss: 0.0311 | Test Acc: 49.56% | LR: 0.001390 | Time: 95.2s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 36.6s
Epoch 141/300 | Loss: 0.0329 | Test Acc: 49.56% | LR: 0.001378 | Time: 95.5s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 36.6s
Epoch 142/300 | Loss: 0.0318 | Test Acc: 49.56% | LR: 0.001367 | Time: 94.7s
  Trained 10000 images, Time: 37.3s
  Trained 20000 images, Time: 37.2s
Epoch 143/300 | Loss: 0.0280 | Test Acc: 49.56% | LR: 0.001355 | Time: 96.5s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.5s
Epoch 144/300 | Loss: 0.0306 | Test Acc: 49.56% | LR: 0.001344 | Time: 93.9s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 36.7s
Epoch 145/300 | Loss: 0.0290 | Test Acc: 49.56% | LR: 0.001332 | Time: 96.0s
  Trained 10000 images, Time: 36.5s
  Trained 20000 images, Time: 36.8s
Epoch 146/300 | Loss: 0.0306 | Test Acc: 49.56% | LR: 0.001321 | Time: 96.1s
  Trained 10000 images, Time: 36.4s
  Trained 20000 images, Time: 36.3s
Epoch 147/300 | Loss: 0.0294 | Test Acc: 49.56% | LR: 0.001309 | Time: 95.3s
  Trained 10000 images, Time: 36.2s
  Trained 20000 images, Time: 36.6s
Epoch 148/300 | Loss: 0.0265 | Test Acc: 49.56% | LR: 0.001297 | Time: 95.4s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.7s
Epoch 149/300 | Loss: 0.0283 | Test Acc: 49.56% | LR: 0.001285 | Time: 94.8s
  Trained 10000 images, Time: 37.1s
  Trained 20000 images, Time: 37.6s
Epoch 150/300 | Loss: 0.0237 | Test Acc: 49.56% | LR: 0.001274 | Time: 97.3s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 37.3s
Epoch 151/300 | Loss: 0.0281 | Test Acc: 49.56% | LR: 0.001262 | Time: 95.5s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 36.7s
Epoch 152/300 | Loss: 0.0251 | Test Acc: 49.56% | LR: 0.001250 | Time: 95.9s
  Trained 10000 images, Time: 36.1s
  Trained 20000 images, Time: 37.3s
Epoch 153/300 | Loss: 0.0252 | Test Acc: 49.56% | LR: 0.001238 | Time: 95.5s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.8s
Epoch 154/300 | Loss: 0.0239 | Test Acc: 49.56% | LR: 0.001226 | Time: 95.5s
  Trained 10000 images, Time: 35.9s
  Trained 20000 images, Time: 36.2s
Epoch 155/300 | Loss: 0.0236 | Test Acc: 49.56% | LR: 0.001214 | Time: 94.4s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.6s
Epoch 156/300 | Loss: 0.0230 | Test Acc: 49.56% | LR: 0.001202 | Time: 95.8s
  Trained 10000 images, Time: 36.5s
  Trained 20000 images, Time: 36.5s
Epoch 157/300 | Loss: 0.0233 | Test Acc: 49.56% | LR: 0.001190 | Time: 95.3s
  Trained 10000 images, Time: 36.5s
  Trained 20000 images, Time: 36.5s
Epoch 158/300 | Loss: 0.0229 | Test Acc: 49.56% | LR: 0.001178 | Time: 94.6s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.9s
Epoch 159/300 | Loss: 0.0239 | Test Acc: 49.56% | LR: 0.001166 | Time: 96.1s
  Trained 10000 images, Time: 37.4s
  Trained 20000 images, Time: 36.7s
Epoch 160/300 | Loss: 0.0224 | Test Acc: 49.56% | LR: 0.001153 | Time: 95.5s
  Trained 10000 images, Time: 37.0s
  Trained 20000 images, Time: 36.9s
Epoch 161/300 | Loss: 0.0220 | Test Acc: 49.56% | LR: 0.001141 | Time: 96.3s
  Trained 10000 images, Time: 37.1s
  Trained 20000 images, Time: 37.0s
Epoch 162/300 | Loss: 0.0231 | Test Acc: 49.56% | LR: 0.001129 | Time: 96.2s
  Trained 10000 images, Time: 37.5s
  Trained 20000 images, Time: 37.1s
Epoch 163/300 | Loss: 0.0224 | Test Acc: 49.56% | LR: 0.001117 | Time: 97.5s
  Trained 10000 images, Time: 36.5s
  Trained 20000 images, Time: 36.4s
Epoch 164/300 | Loss: 0.0203 | Test Acc: 49.56% | LR: 0.001105 | Time: 95.7s
  Trained 10000 images, Time: 36.9s
  Trained 20000 images, Time: 36.9s
Epoch 165/300 | Loss: 0.0219 | Test Acc: 49.56% | LR: 0.001092 | Time: 96.6s
  Trained 10000 images, Time: 37.2s
  Trained 20000 images, Time: 37.9s
Epoch 166/300 | Loss: 0.0198 | Test Acc: 49.56% | LR: 0.001080 | Time: 98.2s
  Trained 10000 images, Time: 36.9s
  Trained 20000 images, Time: 37.2s
Epoch 167/300 | Loss: 0.0183 | Test Acc: 49.56% | LR: 0.001068 | Time: 96.1s
  Trained 10000 images, Time: 36.5s
  Trained 20000 images, Time: 36.8s
Epoch 168/300 | Loss: 0.0205 | Test Acc: 49.56% | LR: 0.001055 | Time: 95.7s
  Trained 10000 images, Time: 36.2s
  Trained 20000 images, Time: 36.6s
Epoch 169/300 | Loss: 0.0177 | Test Acc: 49.56% | LR: 0.001043 | Time: 94.7s
  Trained 10000 images, Time: 37.1s
  Trained 20000 images, Time: 37.1s
Epoch 170/300 | Loss: 0.0196 | Test Acc: 49.56% | LR: 0.001031 | Time: 96.9s
  Trained 10000 images, Time: 35.7s
  Trained 20000 images, Time: 36.4s
Epoch 171/300 | Loss: 0.0192 | Test Acc: 49.56% | LR: 0.001018 | Time: 94.3s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.1s
Epoch 172/300 | Loss: 0.0191 | Test Acc: 49.56% | LR: 0.001006 | Time: 94.6s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.9s
Epoch 173/300 | Loss: 0.0198 | Test Acc: 49.56% | LR: 0.000994 | Time: 95.9s
  Trained 10000 images, Time: 36.4s
  Trained 20000 images, Time: 36.0s
Epoch 174/300 | Loss: 0.0189 | Test Acc: 49.56% | LR: 0.000982 | Time: 93.9s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.4s
Epoch 175/300 | Loss: 0.0172 | Test Acc: 49.56% | LR: 0.000969 | Time: 95.0s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.7s
Epoch 176/300 | Loss: 0.0178 | Test Acc: 49.56% | LR: 0.000957 | Time: 94.9s
  Trained 10000 images, Time: 37.2s
  Trained 20000 images, Time: 37.1s
Epoch 177/300 | Loss: 0.0172 | Test Acc: 49.56% | LR: 0.000945 | Time: 97.2s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 36.6s
Epoch 178/300 | Loss: 0.0180 | Test Acc: 49.56% | LR: 0.000932 | Time: 94.7s
  Trained 10000 images, Time: 36.9s
  Trained 20000 images, Time: 36.6s
Epoch 179/300 | Loss: 0.0155 | Test Acc: 49.56% | LR: 0.000920 | Time: 96.1s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.8s
Epoch 180/300 | Loss: 0.0175 | Test Acc: 49.56% | LR: 0.000908 | Time: 95.1s
  Trained 10000 images, Time: 37.8s
  Trained 20000 images, Time: 40.0s
Epoch 181/300 | Loss: 0.0173 | Test Acc: 49.56% | LR: 0.000895 | Time: 101.0s
  Trained 10000 images, Time: 38.9s
  Trained 20000 images, Time: 40.7s
Epoch 182/300 | Loss: 0.0174 | Test Acc: 49.56% | LR: 0.000883 | Time: 104.4s
  Trained 10000 images, Time: 40.6s
  Trained 20000 images, Time: 43.8s
Epoch 183/300 | Loss: 0.0160 | Test Acc: 49.56% | LR: 0.000871 | Time: 107.9s
  Trained 10000 images, Time: 36.2s
  Trained 20000 images, Time: 36.9s
Epoch 184/300 | Loss: 0.0145 | Test Acc: 49.56% | LR: 0.000859 | Time: 95.2s
  Trained 10000 images, Time: 36.4s
  Trained 20000 images, Time: 36.5s
Epoch 185/300 | Loss: 0.0126 | Test Acc: 49.56% | LR: 0.000847 | Time: 95.2s
  Trained 10000 images, Time: 36.4s
  Trained 20000 images, Time: 36.7s
Epoch 186/300 | Loss: 0.0163 | Test Acc: 49.56% | LR: 0.000834 | Time: 94.7s
  Trained 10000 images, Time: 37.5s
  Trained 20000 images, Time: 37.4s
Epoch 187/300 | Loss: 0.0126 | Test Acc: 49.56% | LR: 0.000822 | Time: 96.9s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.7s
Epoch 188/300 | Loss: 0.0151 | Test Acc: 49.56% | LR: 0.000810 | Time: 94.6s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.5s
Epoch 189/300 | Loss: 0.0132 | Test Acc: 49.56% | LR: 0.000798 | Time: 95.3s
  Trained 10000 images, Time: 36.4s
  Trained 20000 images, Time: 37.0s
Epoch 190/300 | Loss: 0.0123 | Test Acc: 49.56% | LR: 0.000786 | Time: 95.1s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 37.7s
Epoch 191/300 | Loss: 0.0141 | Test Acc: 49.56% | LR: 0.000774 | Time: 96.7s
  Trained 10000 images, Time: 36.0s
  Trained 20000 images, Time: 36.1s
Epoch 192/300 | Loss: 0.0124 | Test Acc: 49.56% | LR: 0.000762 | Time: 94.4s
  Trained 10000 images, Time: 36.2s
  Trained 20000 images, Time: 36.2s
Epoch 193/300 | Loss: 0.0129 | Test Acc: 49.56% | LR: 0.000750 | Time: 95.1s
  Trained 10000 images, Time: 37.0s
  Trained 20000 images, Time: 37.0s
Epoch 194/300 | Loss: 0.0127 | Test Acc: 49.56% | LR: 0.000738 | Time: 96.7s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.5s
Epoch 195/300 | Loss: 0.0125 | Test Acc: 49.56% | LR: 0.000726 | Time: 94.4s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.5s
Epoch 196/300 | Loss: 0.0117 | Test Acc: 49.56% | LR: 0.000715 | Time: 95.2s
  Trained 10000 images, Time: 36.4s
  Trained 20000 images, Time: 37.3s
Epoch 197/300 | Loss: 0.0130 | Test Acc: 49.56% | LR: 0.000703 | Time: 95.5s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 36.7s
Epoch 198/300 | Loss: 0.0122 | Test Acc: 49.56% | LR: 0.000691 | Time: 95.8s
  Trained 10000 images, Time: 36.2s
  Trained 20000 images, Time: 36.7s
Epoch 199/300 | Loss: 0.0120 | Test Acc: 49.56% | LR: 0.000679 | Time: 94.4s
  Trained 10000 images, Time: 36.9s
  Trained 20000 images, Time: 36.7s
Epoch 200/300 | Loss: 0.0100 | Test Acc: 49.56% | LR: 0.000668 | Time: 96.4s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.7s
Epoch 201/300 | Loss: 0.0108 | Test Acc: 49.56% | LR: 0.000656 | Time: 95.2s
  Trained 10000 images, Time: 36.5s
  Trained 20000 images, Time: 36.5s
Epoch 202/300 | Loss: 0.0122 | Test Acc: 49.56% | LR: 0.000645 | Time: 95.3s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.3s
Epoch 203/300 | Loss: 0.0124 | Test Acc: 49.56% | LR: 0.000633 | Time: 95.3s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 37.1s
Epoch 204/300 | Loss: 0.0103 | Test Acc: 49.56% | LR: 0.000622 | Time: 95.7s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 36.5s
Epoch 205/300 | Loss: 0.0114 | Test Acc: 49.56% | LR: 0.000610 | Time: 95.5s
  Trained 10000 images, Time: 36.5s
  Trained 20000 images, Time: 36.4s
Epoch 206/300 | Loss: 0.0100 | Test Acc: 49.56% | LR: 0.000599 | Time: 94.2s
  Trained 10000 images, Time: 36.9s
  Trained 20000 images, Time: 36.6s
Epoch 207/300 | Loss: 0.0086 | Test Acc: 49.56% | LR: 0.000588 | Time: 95.9s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 36.5s
Epoch 208/300 | Loss: 0.0113 | Test Acc: 49.56% | LR: 0.000576 | Time: 94.4s
  Trained 10000 images, Time: 37.0s
  Trained 20000 images, Time: 36.3s
Epoch 209/300 | Loss: 0.0084 | Test Acc: 49.56% | LR: 0.000565 | Time: 95.5s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.5s
Epoch 210/300 | Loss: 0.0096 | Test Acc: 49.56% | LR: 0.000554 | Time: 94.6s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.3s
Epoch 211/300 | Loss: 0.0090 | Test Acc: 49.56% | LR: 0.000543 | Time: 95.5s
  Trained 10000 images, Time: 36.0s
  Trained 20000 images, Time: 36.1s
Epoch 212/300 | Loss: 0.0087 | Test Acc: 49.56% | LR: 0.000532 | Time: 94.6s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 37.0s
Epoch 213/300 | Loss: 0.0076 | Test Acc: 49.56% | LR: 0.000521 | Time: 95.8s
  Trained 10000 images, Time: 36.0s
  Trained 20000 images, Time: 36.5s
Epoch 214/300 | Loss: 0.0088 | Test Acc: 49.56% | LR: 0.000511 | Time: 94.6s
  Trained 10000 images, Time: 36.0s
  Trained 20000 images, Time: 36.8s
Epoch 215/300 | Loss: 0.0091 | Test Acc: 49.56% | LR: 0.000500 | Time: 94.5s
  Trained 10000 images, Time: 36.2s
  Trained 20000 images, Time: 37.3s
Epoch 216/300 | Loss: 0.0091 | Test Acc: 49.56% | LR: 0.000489 | Time: 96.4s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 36.8s
Epoch 217/300 | Loss: 0.0087 | Test Acc: 49.56% | LR: 0.000479 | Time: 95.2s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.7s
Epoch 218/300 | Loss: 0.0095 | Test Acc: 49.56% | LR: 0.000468 | Time: 95.2s
  Trained 10000 images, Time: 36.2s
  Trained 20000 images, Time: 36.8s
Epoch 219/300 | Loss: 0.0081 | Test Acc: 49.56% | LR: 0.000458 | Time: 94.5s
  Trained 10000 images, Time: 37.0s
  Trained 20000 images, Time: 37.3s
Epoch 220/300 | Loss: 0.0082 | Test Acc: 49.56% | LR: 0.000448 | Time: 96.7s
  Trained 10000 images, Time: 36.1s
  Trained 20000 images, Time: 36.2s
Epoch 221/300 | Loss: 0.0089 | Test Acc: 49.56% | LR: 0.000437 | Time: 94.3s
  Trained 10000 images, Time: 37.0s
  Trained 20000 images, Time: 36.8s
Epoch 222/300 | Loss: 0.0082 | Test Acc: 49.56% | LR: 0.000427 | Time: 96.3s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 36.9s
Epoch 223/300 | Loss: 0.0078 | Test Acc: 49.56% | LR: 0.000417 | Time: 96.1s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 36.4s
Epoch 224/300 | Loss: 0.0076 | Test Acc: 49.56% | LR: 0.000407 | Time: 94.7s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 36.5s
Epoch 225/300 | Loss: 0.0070 | Test Acc: 49.56% | LR: 0.000397 | Time: 95.7s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 36.5s
Epoch 226/300 | Loss: 0.0059 | Test Acc: 49.56% | LR: 0.000388 | Time: 94.7s
  Trained 10000 images, Time: 37.5s
  Trained 20000 images, Time: 36.5s
Epoch 227/300 | Loss: 0.0071 | Test Acc: 49.56% | LR: 0.000378 | Time: 96.5s
  Trained 10000 images, Time: 36.5s
  Trained 20000 images, Time: 36.5s
Epoch 228/300 | Loss: 0.0055 | Test Acc: 49.56% | LR: 0.000368 | Time: 94.5s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.7s
Epoch 229/300 | Loss: 0.0063 | Test Acc: 49.56% | LR: 0.000359 | Time: 96.0s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 36.4s
Epoch 230/300 | Loss: 0.0077 | Test Acc: 49.56% | LR: 0.000349 | Time: 95.1s
  Trained 10000 images, Time: 36.1s
  Trained 20000 images, Time: 36.8s
Epoch 231/300 | Loss: 0.0054 | Test Acc: 49.56% | LR: 0.000340 | Time: 95.2s
  Trained 10000 images, Time: 36.0s
  Trained 20000 images, Time: 36.6s
Epoch 232/300 | Loss: 0.0069 | Test Acc: 49.56% | LR: 0.000331 | Time: 94.8s
  Trained 10000 images, Time: 36.4s
  Trained 20000 images, Time: 37.0s
Epoch 233/300 | Loss: 0.0056 | Test Acc: 49.56% | LR: 0.000322 | Time: 95.4s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.7s
Epoch 234/300 | Loss: 0.0050 | Test Acc: 49.56% | LR: 0.000313 | Time: 95.7s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 37.2s
Epoch 235/300 | Loss: 0.0057 | Test Acc: 49.56% | LR: 0.000304 | Time: 95.6s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 37.6s
Epoch 236/300 | Loss: 0.0069 | Test Acc: 49.56% | LR: 0.000295 | Time: 97.7s
  Trained 10000 images, Time: 37.2s
  Trained 20000 images, Time: 37.1s
Epoch 237/300 | Loss: 0.0040 | Test Acc: 49.56% | LR: 0.000286 | Time: 95.6s
  Trained 10000 images, Time: 37.0s
  Trained 20000 images, Time: 36.8s
Epoch 238/300 | Loss: 0.0043 | Test Acc: 49.56% | LR: 0.000278 | Time: 96.4s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.6s
Epoch 239/300 | Loss: 0.0068 | Test Acc: 49.56% | LR: 0.000269 | Time: 95.1s
  Trained 10000 images, Time: 37.3s
  Trained 20000 images, Time: 36.8s
Epoch 240/300 | Loss: 0.0061 | Test Acc: 49.56% | LR: 0.000261 | Time: 96.9s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.9s
Epoch 241/300 | Loss: 0.0052 | Test Acc: 49.56% | LR: 0.000253 | Time: 96.1s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 36.5s
Epoch 242/300 | Loss: 0.0057 | Test Acc: 49.56% | LR: 0.000245 | Time: 95.0s
  Trained 10000 images, Time: 37.1s
  Trained 20000 images, Time: 37.3s
Epoch 243/300 | Loss: 0.0049 | Test Acc: 49.56% | LR: 0.000237 | Time: 97.5s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 36.7s
Epoch 244/300 | Loss: 0.0051 | Test Acc: 49.56% | LR: 0.000229 | Time: 95.6s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 36.6s
Epoch 245/300 | Loss: 0.0056 | Test Acc: 49.56% | LR: 0.000221 | Time: 95.9s
  Trained 10000 images, Time: 36.2s
  Trained 20000 images, Time: 37.1s
Epoch 246/300 | Loss: 0.0040 | Test Acc: 49.56% | LR: 0.000213 | Time: 96.0s
  Trained 10000 images, Time: 37.0s
  Trained 20000 images, Time: 37.3s
Epoch 247/300 | Loss: 0.0040 | Test Acc: 49.56% | LR: 0.000206 | Time: 97.1s
  Trained 10000 images, Time: 36.1s
  Trained 20000 images, Time: 36.8s
Epoch 248/300 | Loss: 0.0047 | Test Acc: 49.56% | LR: 0.000198 | Time: 95.4s
  Trained 10000 images, Time: 36.1s
  Trained 20000 images, Time: 37.0s
Epoch 249/300 | Loss: 0.0059 | Test Acc: 49.56% | LR: 0.000191 | Time: 95.4s
  Trained 10000 images, Time: 36.9s
  Trained 20000 images, Time: 36.8s
Epoch 250/300 | Loss: 0.0032 | Test Acc: 49.56% | LR: 0.000184 | Time: 96.5s
  Trained 10000 images, Time: 36.4s
  Trained 20000 images, Time: 37.0s
Epoch 251/300 | Loss: 0.0051 | Test Acc: 49.56% | LR: 0.000177 | Time: 95.0s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 36.9s
Epoch 252/300 | Loss: 0.0042 | Test Acc: 49.56% | LR: 0.000170 | Time: 96.6s
  Trained 10000 images, Time: 37.2s
  Trained 20000 images, Time: 36.6s
Epoch 253/300 | Loss: 0.0045 | Test Acc: 49.56% | LR: 0.000163 | Time: 95.2s
  Trained 10000 images, Time: 37.0s
  Trained 20000 images, Time: 37.0s
Epoch 254/300 | Loss: 0.0035 | Test Acc: 49.56% | LR: 0.000156 | Time: 96.3s
  Trained 10000 images, Time: 36.5s
  Trained 20000 images, Time: 36.2s
Epoch 255/300 | Loss: 0.0039 | Test Acc: 49.56% | LR: 0.000150 | Time: 95.0s
  Trained 10000 images, Time: 37.1s
  Trained 20000 images, Time: 36.6s
Epoch 256/300 | Loss: 0.0039 | Test Acc: 49.56% | LR: 0.000143 | Time: 95.9s
  Trained 10000 images, Time: 36.5s
  Trained 20000 images, Time: 36.8s
Epoch 257/300 | Loss: 0.0037 | Test Acc: 49.56% | LR: 0.000137 | Time: 95.7s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 36.7s
Epoch 258/300 | Loss: 0.0039 | Test Acc: 49.56% | LR: 0.000131 | Time: 95.0s
  Trained 10000 images, Time: 36.9s
  Trained 20000 images, Time: 37.1s
Epoch 259/300 | Loss: 0.0028 | Test Acc: 49.56% | LR: 0.000125 | Time: 96.6s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.6s
Epoch 260/300 | Loss: 0.0024 | Test Acc: 49.56% | LR: 0.000119 | Time: 94.5s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.8s
Epoch 261/300 | Loss: 0.0037 | Test Acc: 49.56% | LR: 0.000113 | Time: 96.8s
  Trained 10000 images, Time: 37.4s
  Trained 20000 images, Time: 38.1s
Epoch 262/300 | Loss: 0.0040 | Test Acc: 49.56% | LR: 0.000108 | Time: 97.5s
  Trained 10000 images, Time: 37.3s
  Trained 20000 images, Time: 37.6s
Epoch 263/300 | Loss: 0.0033 | Test Acc: 49.56% | LR: 0.000102 | Time: 97.9s
  Trained 10000 images, Time: 35.9s
  Trained 20000 images, Time: 36.3s
Epoch 264/300 | Loss: 0.0043 | Test Acc: 49.56% | LR: 0.000097 | Time: 94.4s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 37.8s
Epoch 265/300 | Loss: 0.0031 | Test Acc: 49.56% | LR: 0.000092 | Time: 97.8s
  Trained 10000 images, Time: 36.9s
  Trained 20000 images, Time: 37.1s
Epoch 266/300 | Loss: 0.0021 | Test Acc: 49.56% | LR: 0.000086 | Time: 96.4s
  Trained 10000 images, Time: 36.2s
  Trained 20000 images, Time: 37.0s
Epoch 267/300 | Loss: 0.0033 | Test Acc: 49.56% | LR: 0.000082 | Time: 94.7s
  Trained 10000 images, Time: 36.4s
  Trained 20000 images, Time: 36.8s
Epoch 268/300 | Loss: 0.0026 | Test Acc: 49.56% | LR: 0.000077 | Time: 95.5s
  Trained 10000 images, Time: 37.0s
  Trained 20000 images, Time: 37.4s
Epoch 269/300 | Loss: 0.0032 | Test Acc: 49.56% | LR: 0.000072 | Time: 95.9s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 36.4s
Epoch 270/300 | Loss: 0.0021 | Test Acc: 49.56% | LR: 0.000068 | Time: 95.2s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 36.4s
Epoch 271/300 | Loss: 0.0026 | Test Acc: 49.56% | LR: 0.000063 | Time: 94.7s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 37.0s
Epoch 272/300 | Loss: 0.0040 | Test Acc: 49.56% | LR: 0.000059 | Time: 96.4s
  Trained 10000 images, Time: 36.3s
  Trained 20000 images, Time: 37.2s
Epoch 273/300 | Loss: 0.0039 | Test Acc: 49.56% | LR: 0.000055 | Time: 96.0s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 36.2s
Epoch 274/300 | Loss: 0.0020 | Test Acc: 49.56% | LR: 0.000051 | Time: 94.5s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 37.0s
Epoch 275/300 | Loss: 0.0027 | Test Acc: 49.56% | LR: 0.000047 | Time: 97.1s
  Trained 10000 images, Time: 37.4s
  Trained 20000 images, Time: 36.8s
Epoch 276/300 | Loss: 0.0041 | Test Acc: 49.56% | LR: 0.000043 | Time: 95.6s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.7s
Epoch 277/300 | Loss: 0.0025 | Test Acc: 49.56% | LR: 0.000040 | Time: 96.1s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.9s
Epoch 278/300 | Loss: 0.0032 | Test Acc: 49.56% | LR: 0.000037 | Time: 95.0s
  Trained 10000 images, Time: 36.7s
  Trained 20000 images, Time: 36.8s
Epoch 279/300 | Loss: 0.0029 | Test Acc: 49.56% | LR: 0.000033 | Time: 96.1s
  Trained 10000 images, Time: 36.2s
  Trained 20000 images, Time: 36.6s
Epoch 280/300 | Loss: 0.0027 | Test Acc: 49.56% | LR: 0.000030 | Time: 94.7s
  Trained 10000 images, Time: 36.1s
  Trained 20000 images, Time: 36.5s
Epoch 281/300 | Loss: 0.0025 | Test Acc: 49.56% | LR: 0.000027 | Time: 95.4s
  Trained 10000 images, Time: 36.5s
  Trained 20000 images, Time: 37.1s
Epoch 282/300 | Loss: 0.0026 | Test Acc: 49.56% | LR: 0.000024 | Time: 96.1s
  Trained 10000 images, Time: 36.0s
  Trained 20000 images, Time: 36.5s
Epoch 283/300 | Loss: 0.0022 | Test Acc: 49.56% | LR: 0.000022 | Time: 94.0s
  Trained 10000 images, Time: 36.4s
  Trained 20000 images, Time: 37.0s
Epoch 284/300 | Loss: 0.0032 | Test Acc: 49.56% | LR: 0.000019 | Time: 95.7s
  Trained 10000 images, Time: 36.4s
  Trained 20000 images, Time: 36.8s
Epoch 285/300 | Loss: 0.0016 | Test Acc: 49.56% | LR: 0.000017 | Time: 94.9s
  Trained 10000 images, Time: 37.3s
  Trained 20000 images, Time: 36.5s
Epoch 286/300 | Loss: 0.0026 | Test Acc: 49.56% | LR: 0.000015 | Time: 96.4s
  Trained 10000 images, Time: 37.2s
  Trained 20000 images, Time: 37.1s
Epoch 287/300 | Loss: 0.0022 | Test Acc: 49.56% | LR: 0.000013 | Time: 95.7s
  Trained 10000 images, Time: 37.6s
  Trained 20000 images, Time: 37.4s
Epoch 288/300 | Loss: 0.0029 | Test Acc: 49.56% | LR: 0.000011 | Time: 97.6s
  Trained 10000 images, Time: 37.3s
  Trained 20000 images, Time: 37.2s
Epoch 289/300 | Loss: 0.0022 | Test Acc: 49.56% | LR: 0.000009 | Time: 96.6s
  Trained 10000 images, Time: 37.3s
  Trained 20000 images, Time: 36.9s
Epoch 290/300 | Loss: 0.0019 | Test Acc: 49.56% | LR: 0.000008 | Time: 96.9s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 36.8s
Epoch 291/300 | Loss: 0.0022 | Test Acc: 49.56% | LR: 0.000006 | Time: 96.4s
  Trained 10000 images, Time: 36.6s
  Trained 20000 images, Time: 36.9s
Epoch 292/300 | Loss: 0.0021 | Test Acc: 49.56% | LR: 0.000005 | Time: 96.1s
  Trained 10000 images, Time: 37.6s
  Trained 20000 images, Time: 37.6s
Epoch 293/300 | Loss: 0.0024 | Test Acc: 49.56% | LR: 0.000004 | Time: 98.2s
  Trained 10000 images, Time: 36.8s
  Trained 20000 images, Time: 37.3s
Epoch 294/300 | Loss: 0.0033 | Test Acc: 49.56% | LR: 0.000003 | Time: 96.1s
  Trained 10000 images, Time: 37.1s
  Trained 20000 images, Time: 37.1s
Epoch 295/300 | Loss: 0.0018 | Test Acc: 49.56% | LR: 0.000002 | Time: 96.5s
  Trained 10000 images, Time: 35.9s
  Trained 20000 images, Time: 36.7s
Epoch 296/300 | Loss: 0.0020 | Test Acc: 49.56% | LR: 0.000001 | Time: 95.3s
  Trained 10000 images, Time: 36.5s
  Trained 20000 images, Time: 36.4s
Epoch 297/300 | Loss: 0.0025 | Test Acc: 49.56% | LR: 0.000001 | Time: 95.0s
  Trained 10000 images, Time: 36.0s
  Trained 20000 images, Time: 36.7s
Epoch 298/300 | Loss: 0.0026 | Test Acc: 49.56% | LR: 0.000000 | Time: 94.9s
  Trained 10000 images, Time: 36.1s
  Trained 20000 images, Time: 36.6s
Epoch 299/300 | Loss: 0.0023 | Test Acc: 49.56% | LR: 0.000000 | Time: 95.0s
  Trained 10000 images, Time: 36.9s
  Trained 20000 images, Time: 37.1s
Epoch 300/300 | Loss: 0.0030 | Test Acc: 49.56% | LR: 0.000000 | Time: 96.3s

Finished Training in 28799.6 seconds
Best Test Accuracy: 76.31%


rancom logs
Epoch 01/250 | Loss: 0.6862 | Test Accuracy: 57.40% | Time: 66.6s
Epoch 02/250 | Loss: 0.6744 | Test Accuracy: 62.17% | Time: 66.6s
Epoch 03/250 | Loss: 0.6394 | Test Accuracy: 65.02% | Time: 66.2s
Epoch 04/250 | Loss: 0.6356 | Test Accuracy: 65.32% | Time: 67.1s
Epoch 05/250 | Loss: 0.6179 | Test Accuracy: 66.23% | Time: 67.2s
Epoch 06/250 | Loss: 0.6061 | Test Accuracy: 67.63% | Time: 66.7s
Epoch 07/250 | Loss: 0.5994 | Test Accuracy: 70.56% | Time: 66.7s
Epoch 08/250 | Loss: 0.5905 | Test Accuracy: 70.79% | Time: 67.2s
Epoch 09/250 | Loss: 0.5848 | Test Accuracy: 70.06% | Time: 66.2s
Epoch 10/250 | Loss: 0.5828 | Test Accuracy: 70.80% | Time: 67.2s
Epoch 11/250 | Loss: 0.5777 | Test Accuracy: 69.32% | Time: 67.1s
Epoch 12/250 | Loss: 0.5770 | Test Accuracy: 67.86% | Time: 66.3s
Epoch 13/250 | Loss: 0.5713 | Test Accuracy: 71.04% | Time: 66.7s
Epoch 14/250 | Loss: 0.5733 | Test Accuracy: 71.16% | Time: 66.3s
Epoch 15/250 | Loss: 0.5676 | Test Accuracy: 70.10% | Time: 65.9s
Epoch 16/250 | Loss: 0.5611 | Test Accuracy: 68.52% | Time: 66.9s
Epoch 17/250 | Loss: 0.5581 | Test Accuracy: 62.66% | Time: 66.7s
Epoch 18/250 | Loss: 0.5549 | Test Accuracy: 69.12% | Time: 65.9s
Epoch 19/250 | Loss: 0.5475 | Test Accuracy: 68.78% | Time: 66.1s
Epoch 20/250 | Loss: 0.5475 | Test Accuracy: 67.81% | Time: 67.3s
Epoch 21/250 | Loss: 0.5429 | Test Accuracy: 69.73% | Time: 66.1s
Epoch 22/250 | Loss: 0.5434 | Test Accuracy: 70.94% | Time: 66.1s
Epoch 23/250 | Loss: 0.5314 | Test Accuracy: 70.03% | Time: 65.9s
Epoch 24/250 | Loss: 0.5272 | Test Accuracy: 70.51% | Time: 66.4s
Epoch 25/250 | Loss: 0.5200 | Test Accuracy: 69.18% | Time: 67.0s
Epoch 26/250 | Loss: 0.5197 | Test Accuracy: 69.06% | Time: 66.3s
Epoch 27/250 | Loss: 0.5092 | Test Accuracy: 70.29% | Time: 67.1s
Epoch 28/250 | Loss: 0.5045 | Test Accuracy: 68.53% | Time: 66.6s
Epoch 29/250 | Loss: 0.4973 | Test Accuracy: 67.89% | Time: 66.0s
Epoch 30/250 | Loss: 0.4978 | Test Accuracy: 72.46% | Time: 66.9s
Epoch 31/250 | Loss: 0.4924 | Test Accuracy: 69.01% | Time: 66.8s
Epoch 32/250 | Loss: 0.4821 | Test Accuracy: 71.88% | Time: 66.9s
Epoch 33/250 | Loss: 0.4801 | Test Accuracy: 71.88% | Time: 65.9s
Epoch 34/250 | Loss: 0.4821 | Test Accuracy: 72.58% | Time: 66.6s
Epoch 35/250 | Loss: 0.4754 | Test Accuracy: 72.00% | Time: 66.2s
Epoch 36/250 | Loss: 0.4642 | Test Accuracy: 72.01% | Time: 66.1s
Epoch 37/250 | Loss: 0.4707 | Test Accuracy: 70.86% | Time: 66.0s
Epoch 38/250 | Loss: 0.4619 | Test Accuracy: 73.21% | Time: 67.5s
Epoch 39/250 | Loss: 0.4584 | Test Accuracy: 70.82% | Time: 66.2s
Epoch 40/250 | Loss: 0.4588 | Test Accuracy: 71.47% | Time: 66.1s
Epoch 41/250 | Loss: 0.4503 | Test Accuracy: 69.36% | Time: 66.2s
Epoch 42/250 | Loss: 0.4487 | Test Accuracy: 73.92% | Time: 66.1s
Epoch 43/250 | Loss: 0.4471 | Test Accuracy: 73.66% | Time: 66.1s
Epoch 44/250 | Loss: 0.4420 | Test Accuracy: 70.40% | Time: 69.1s
Epoch 45/250 | Loss: 0.4372 | Test Accuracy: 74.51% | Time: 65.8s
Epoch 46/250 | Loss: 0.4275 | Test Accuracy: 73.72% | Time: 66.2s
Epoch 47/250 | Loss: 0.4372 | Test Accuracy: 71.18% | Time: 66.1s
Epoch 48/250 | Loss: 0.4249 | Test Accuracy: 73.68% | Time: 66.4s
Epoch 49/250 | Loss: 0.4201 | Test Accuracy: 73.56% | Time: 66.4s
Epoch 50/250 | Loss: 0.4210 | Test Accuracy: 71.82% | Time: 66.5s
Epoch 51/250 | Loss: 0.4220 | Test Accuracy: 74.36% | Time: 67.2s
Epoch 52/250 | Loss: 0.4170 | Test Accuracy: 74.48% | Time: 66.1s
Epoch 53/250 | Loss: 0.4128 | Test Accuracy: 72.56% | Time: 66.1s
Epoch 54/250 | Loss: 0.4098 | Test Accuracy: 73.81% | Time: 66.1s
Epoch 55/250 | Loss: 0.4019 | Test Accuracy: 72.63% | Time: 66.1s
Epoch 56/250 | Loss: 0.4017 | Test Accuracy: 75.47% | Time: 66.4s
Epoch 57/250 | Loss: 0.3993 | Test Accuracy: 73.22% | Time: 66.8s
Epoch 58/250 | Loss: 0.3999 | Test Accuracy: 74.11% | Time: 66.9s
Epoch 59/250 | Loss: 0.3910 | Test Accuracy: 74.24% | Time: 66.5s
Epoch 60/250 | Loss: 0.3946 | Test Accuracy: 71.89% | Time: 65.9s
Epoch 61/250 | Loss: 0.3929 | Test Accuracy: 73.89% | Time: 67.6s
Epoch 62/250 | Loss: 0.3861 | Test Accuracy: 72.26% | Time: 66.3s
Epoch 63/250 | Loss: 0.3853 | Test Accuracy: 75.48% | Time: 65.9s
Epoch 64/250 | Loss: 0.3843 | Test Accuracy: 75.27% | Time: 65.9s
Epoch 65/250 | Loss: 0.3762 | Test Accuracy: 74.56% | Time: 65.9s
Epoch 66/250 | Loss: 0.3741 | Test Accuracy: 71.28% | Time: 66.2s
Epoch 67/250 | Loss: 0.3777 | Test Accuracy: 75.30% | Time: 66.1s
Epoch 68/250 | Loss: 0.3707 | Test Accuracy: 73.10% | Time: 66.8s
Epoch 69/250 | Loss: 0.3694 | Test Accuracy: 74.92% | Time: 66.6s
Epoch 70/250 | Loss: 0.3723 | Test Accuracy: 74.47% | Time: 66.6s
Epoch 71/250 | Loss: 0.3617 | Test Accuracy: 73.23% | Time: 66.2s
Epoch 72/250 | Loss: 0.3662 | Test Accuracy: 75.86% | Time: 66.6s
Epoch 73/250 | Loss: 0.3605 | Test Accuracy: 73.82% | Time: 66.4s
Epoch 74/250 | Loss: 0.3557 | Test Accuracy: 74.08% | Time: 66.8s
Epoch 75/250 | Loss: 0.3551 | Test Accuracy: 76.37% | Time: 67.3s
Epoch 76/250 | Loss: 0.3572 | Test Accuracy: 70.10% | Time: 66.7s
Epoch 77/250 | Loss: 0.3581 | Test Accuracy: 74.30% | Time: 66.4s
Epoch 78/250 | Loss: 0.3447 | Test Accuracy: 73.14% | Time: 66.2s
Epoch 79/250 | Loss: 0.3417 | Test Accuracy: 75.10% | Time: 66.1s
Epoch 80/250 | Loss: 0.3451 | Test Accuracy: 76.31% | Time: 66.1s
Epoch 81/250 | Loss: 0.3459 | Test Accuracy: 75.82% | Time: 65.9s
Epoch 82/250 | Loss: 0.3380 | Test Accuracy: 75.08% | Time: 66.2s
Epoch 83/250 | Loss: 0.3372 | Test Accuracy: 75.86% | Time: 66.2s
Epoch 84/250 | Loss: 0.3356 | Test Accuracy: 73.20% | Time: 67.6s
Epoch 85/250 | Loss: 0.3431 | Test Accuracy: 74.06% | Time: 66.5s
Epoch 86/250 | Loss: 0.3343 | Test Accuracy: 75.83% | Time: 66.4s
Epoch 87/250 | Loss: 0.3360 | Test Accuracy: 76.71% | Time: 66.1s
Epoch 88/250 | Loss: 0.3335 | Test Accuracy: 73.74% | Time: 66.7s
Epoch 89/250 | Loss: 0.3255 | Test Accuracy: 75.71% | Time: 66.1s
Epoch 90/250 | Loss: 0.3282 | Test Accuracy: 76.62% | Time: 67.6s
Epoch 91/250 | Loss: 0.3216 | Test Accuracy: 74.96% | Time: 67.3s
Epoch 92/250 | Loss: 0.3245 | Test Accuracy: 76.29% | Time: 66.2s
Epoch 93/250 | Loss: 0.3220 | Test Accuracy: 76.04% | Time: 66.4s

0.7425
0.6233
0.6200
0.6060
0.5631
0.5381
0.5537
0.5433
0.4914
0.4871
0.5148
0.4695
0.4381
0.4716
0.4712
0.4199
0.4282
0.4514
0.4230
0.3927

## PREV MODEL


In [ ]:
class MiniConvNeXt(nn.Module):
    def __init__(self, in_chans=1, num_classes=2,
                 depths=(2, 2, 6, 2), dims=(48, 96, 192, 384),
                 layer_scale_init_value=1e-5, drop_path_rate=0.1):
        super().__init__()
        assert len(depths) == 4 and len(dims) == 4

        self.dropout = nn.Dropout(p=0.2)

        # Stem
        self.downsamples = nn.ModuleList()
        stem = nn.Sequential(
            nn.Conv2d(in_chans, dims[0], kernel_size=4, stride=4),
        )
        self.downsamples.append(stem)

        # Downsampling layers between stages
        for i in range(3):
            self.downsamples.append(
                nn.Sequential(
                    nn.Conv2d(dims[i], dims[i+1], kernel_size=2, stride=2)
                )
            )

        # Calculate stochastic depth rates (linearly increasing)
        dp_rates = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]

        # Build stages with progressive drop path
        self.stages = nn.ModuleList()
        cur = 0
        for i in range(4):
            blocks = []
            for _ in range(depths[i]):
                blocks.append(SmallBlock(dims[i],
                                        layer_scale_init_value=layer_scale_init_value,
                                        drop_path=dp_rates[cur]))
                cur += 1
            self.stages.append(nn.Sequential(*blocks))

        # Final norm and head
        self.final_norm = nn.LayerNorm(dims[-1], eps=1e-6)
        self.head = nn.Linear(dims[-1], num_classes)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                trunc_normal_(m.weight, std=.02)
                if getattr(m, "bias", None) is not None:
                    nn.init.constant_(m.bias, 0)

    def forward_features(self, x):
        # x: N, C, H, W  (C=1)
        for i in range(4):
            x = self.downsamples[i](x)
            x = self.stages[i](x)
        # Global average pooling
        x = x.mean([-2, -1])            # N, C
        x = self.final_norm(x)
        return x

    def forward(self, x):
        x = self.forward_features(x)
        x = self.dropout(x)
        x = self.head(x)
        return x

In [ ]:
0.000127
0.000264
0.000476
0.000744
0.001041
0.001338
0.001605
0.001817
0.001954
0.002000
0.002000
0.001999
0.001999
0.001998
0.001997
0.001995
0.001993
0.001991
0.001989
0.001986
0.001983
0.001980
0.001977
0.001973
0.001969
0.001965
0.001961
0.001956
0.001951
0.001946
0.001940
0.001935
0.001929
0.001922
0.001916
0.001909
0.001902
0.001895
0.001887
0.001879
0.001871
0.001863
0.001855
0.001846
0.001837
0.001828
0.001819
0.001809
0.001799
0.001789
0.001779
0.001768
0.001758
0.001747
0.001736
0.001724
0.001713
0.001701
0.001689
0.001677
0.001665
0.001653
0.001640
0.001627
0.001614
0.001601
0.001588
0.001574
0.001561
0.001547
0.001533
0.001519
0.001505
0.001490
0.001476
0.001461
0.001447
0.001432
0.001417
0.001402
0.001386
0.001371
0.001356
0.001340
0.001325
0.001309
0.001293
0.001277
0.001261
0.001245
0.001229
0.001213
0.001197
0.001181
0.001165
0.001148
0.001132
0.001115
0.001099
0.001082
0.001066
0.001049
0.001033
0.001016
0.001000
0.000983
0.000967
0.000950
0.000934
0.000917
0.000901
0.000884
0.000868
0.000852
0.000835
0.000819
0.000803
0.000787
0.000770
0.000754
0.000738
0.000723
0.000707
0.000691
0.000675
0.000660
0.000644
0.000629
0.000613
0.000598
0.000583
0.000568
0.000553
0.000539
0.000524
0.000509
0.000495
0.000481
0.000467
0.000453
0.000439
0.000426
0.000412
0.000399
0.000386
0.000373
0.000360
0.000347
0.000335
0.000323
0.000311
0.000299
0.000287
0.000276
0.000264
0.000253
0.000242
0.000232
0.000221
0.000211
0.000201
0.000191
0.000181
0.000172
0.000163
0.000154
0.000145
0.000137
0.000128
0.000120
0.000113
0.000105
0.000098
0.000091
0.000084
0.000078
0.000071
0.000065
0.000060
0.000054
0.000049
0.000044
0.000039
0.000035
0.000031
0.000027
0.000023
0.000020
0.000016
0.000014
0.000011
0.000009